## Cell 1 — Library Install
Library for pull data through web API


In [ ]:
!pip install -q pandas requests python-dotenv sqlalchemy psycopg2-binary

## Cell 2 — Import Libraries and Initialize the Notebook


In [ ]:
import os
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone

print("Environment ready!")

Environment ready!


## Cell 3 — ADZUNA_APP_ID = getpass("Enter your Adzuna App ID: ")


In [ ]:
from getpass import getpass

ADZUNA_APP_ID = getpass("Enter your Adzuna App ID: ")
ADZUNA_APP_KEY = getpass("Enter your Adzuna App Key: ")

print("Credentials loaded securely.")

Enter your Adzuna App ID: ··········
Enter your Adzuna App Key: ··········
Credentials loaded securely.


## Cell 4 — url = "https://api.adzuna.com/v1/api/jobs/in/search/1"


In [ ]:
import requests
import time

url = "https://api.adzuna.com/v1/api/jobs/in/search/1"

params = {
    "app_id": ADZUNA_APP_ID,
    "app_key": ADZUNA_APP_KEY,
    "results_per_page": 5,
    "what": "data analyst",
    "where": "India"
}

for attempt in range(3):
    print(f"Attempt {attempt + 1}...")

    response = requests.get(
        url,
        params=params,
        headers={
            "User-Agent": "Job-Market-Intelligence-Portfolio/1.0"
        },
        timeout=30
    )

    print("Status:", response.status_code)

    if response.status_code == 200:
        print("✅ Adzuna API is responding!")
        data = response.json()
        print("Jobs found:", data.get("count"))
        break

    elif response.status_code == 503:
        print("⚠️ Adzuna server is temporarily unavailable.")
        time.sleep(5)

    else:
        print(response.text[:500])
        break

Attempt 1...
Status: 200
✅ Adzuna API is responding!
Jobs found: 3432


## Cell 5 — API CONFIGURATION


In [ ]:
import requests
import pandas as pd
import json
import os
from datetime import datetime, timezone

# ==============================
# API CONFIGURATION
# ==============================

BASE_URL = "https://api.adzuna.com/v1/api/jobs/in/search"

SEARCH_QUERIES = [
    "data analyst",
    "business analyst",
    "BI analyst",
    "data science",
    "python developer"
]

LOCATION = "India"
RESULTS_PER_PAGE = 50

# ==============================
# STORAGE
# ==============================

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

all_jobs = []

# ==============================
# FETCH JOBS
# ==============================

for query in SEARCH_QUERIES:

    print(f"\n🔎 Searching: {query}")

    url = f"{BASE_URL}/1"

    params = {
        "app_id": ADZUNA_APP_ID,
        "app_key": ADZUNA_APP_KEY,
        "results_per_page": RESULTS_PER_PAGE,
        "what": query,
        "where": LOCATION
    }

    response = requests.get(
        url,
        params=params,
        headers={
            "User-Agent": "Job-Market-Intelligence-Portfolio/1.0"
        },
        timeout=30
    )

    print("Status:", response.status_code)

    if response.status_code == 200:

        data = response.json()

        jobs = data.get("results", [])

        print(f"✅ Jobs received: {len(jobs)}")

        for job in jobs:

            job["search_query"] = query
            job["ingested_at"] = datetime.now(timezone.utc).isoformat()

            all_jobs.append(job)

    else:
        print("❌ Request failed")
        print(response.text[:300])


# ==============================
# CREATE DATAFRAME
# ==============================

jobs_df = pd.json_normalize(all_jobs)

print("\n==============================")
print("INGESTION COMPLETE")
print("==============================")

print("Total records:", len(jobs_df))
print("Columns:", len(jobs_df.columns))

display(jobs_df.head())


🔎 Searching: data analyst
Status: 200
✅ Jobs received: 50

🔎 Searching: business analyst
Status: 200
✅ Jobs received: 50

🔎 Searching: BI analyst
Status: 200
✅ Jobs received: 50

🔎 Searching: data science
Status: 200
✅ Jobs received: 50

🔎 Searching: python developer
Status: 200
✅ Jobs received: 50

INGESTION COMPLETE
Total records: 250
Columns: 24


,__CLASS__,created,salary_min,redirect_url,salary_max,description,salary_is_predicted,title,contract_time,id,...,company.display_name,company.__CLASS__,location.area,location.__CLASS__,location.display_name,category.__CLASS__,category.label,category.tag,latitude,longitude
0,Adzuna::API::Response::Job,2026-08-03T06:51:53Z,264000.0,https://www.adzuna.in/details/5826247065?utm_m...,276000.0,Data Analyst (Full-Time) Company: Inficore Sof...,0,Data Analyst,full_time,5826247065,...,Inficore Soft,Adzuna::API::Response::Company,[India],Adzuna::API::Response::Location,India,Adzuna::API::Response::Category,IT Jobs,it-jobs,NaN,NaN
1,Adzuna::API::Response::Job,2024-05-10T07:38:46Z,NaN,https://www.adzuna.in/details/4686704238?utm_m...,NaN,Immediate requirement for the post of Data Ana...,0,Data Analyst,NaN,4686704238,...,NaN,Adzuna::API::Response::Company,"[India, Kerala, Thiruvananthapuram]",Adzuna::API::Response::Location,"Thiruvananthapuram, Kerala",Adzuna::API::Response::Category,IT Jobs,it-jobs,8.38159,77.00370
2,Adzuna::API::Response::Job,2026-09-26T07:08:57Z,500000.0,https://www.adzuna.in/details/5899028729?utm_m...,600000.0,Job Summary We are seeking a detail-oriented a...,0,Data Analyst,full_time,5899028729,...,ALTRAIZE,Adzuna::API::Response::Company,"[India, Karnataka, Bangalore]",Adzuna::API::Response::Location,"Bangalore, Karnataka",Adzuna::API::Response::Category,IT Jobs,it-jobs,NaN,NaN
3,Adzuna::API::Response::Job,2026-08-20T07:10:41Z,600000.0,https://www.adzuna.in/details/5849116040?utm_m...,1300000.0,Role: Data Analyst / Analytics Engineer About ...,0,Data Analyst,full_time,5849116040,...,Time Hack Consulting,Adzuna::API::Response::Company,"[India, Karnataka, Bangalore]",Adzuna::API::Response::Location,"Bangalore, Karnataka",Adzuna::API::Response::Category,IT Jobs,it-jobs,NaN,NaN
4,Adzuna::API::Response::Job,2026-09-29T07:58:49Z,800000.0,https://www.adzuna.in/details/5902287266?utm_m...,1000000.0,"Role Overview: As a Data Analyst, you turn cam...",0,Data Analyst,full_time,5902287266,...,Rekruton Technologies,Adzuna::API::Response::Company,"[India, Gujarat, Ahmedabad]",Adzuna::API::Response::Location,"Ahmedabad, Gujarat",Adzuna::API::Response::Category,IT Jobs,it-jobs,23.04200,72.59465


## Cell 6 — raw_file


In [ ]:
raw_file = "data/raw/adzuna_jobs_raw.json"

with open(raw_file, "w", encoding="utf-8") as f:
    json.dump(all_jobs, f, ensure_ascii=False, indent=2)

print(f"✅ Raw data saved: {raw_file}")

✅ Raw data saved: data/raw/adzuna_jobs_raw.json


## Cell 7 — processed_file


In [ ]:
processed_file = "data/processed/jobs_initial.csv"

jobs_df.to_csv(processed_file, index=False)

print(f"✅ Dataset saved: {processed_file}")

✅ Dataset saved: data/processed/jobs_initial.csv


**STEP 2 — CLEAN & STANDARDIZE JOB DATA**

## Cell 9 — STEP 2: CLEAN & STANDARDIZE JOB DATA


In [ ]:
# ============================================
# STEP 2 — CLEAN & STANDARDIZE JOB DATA
# ============================================

import pandas as pd
import numpy as np
import re

# Work on a copy
jobs_clean = jobs_df.copy()

print("Original records:", len(jobs_clean))


# --------------------------------------------
# 1. Rename important columns
# --------------------------------------------

column_mapping = {
    "id": "job_id",
    "title": "job_title",
    "description": "description",
    "created": "posted_date",
    "redirect_url": "job_url",
    "company.display_name": "company",
    "location.display_name": "location",
    "salary_min": "salary_min",
    "salary_max": "salary_max",
    "category.label": "category"
}

jobs_clean = jobs_clean.rename(columns=column_mapping)


# --------------------------------------------
# 2. Keep only useful columns
# --------------------------------------------

required_columns = [
    "job_id",
    "job_title",
    "company",
    "location",
    "description",
    "posted_date",
    "salary_min",
    "salary_max",
    "job_url",
    "category",
    "search_query",
    "ingested_at"
]

# Only keep columns that actually exist
available_columns = [
    col for col in required_columns
    if col in jobs_clean.columns
]

jobs_clean = jobs_clean[available_columns]


# --------------------------------------------
# 3. Remove duplicate jobs
# --------------------------------------------

before = len(jobs_clean)

if "job_id" in jobs_clean.columns:
    jobs_clean = jobs_clean.drop_duplicates(
        subset=["job_id"]
    )

after = len(jobs_clean)

print("Duplicates removed:", before - after)


# --------------------------------------------
# 4. Clean text fields
# --------------------------------------------

text_columns = [
    "job_title",
    "company",
    "location",
    "category",
    "search_query"
]

for col in text_columns:

    if col in jobs_clean.columns:

        jobs_clean[col] = (
            jobs_clean[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )


# --------------------------------------------
# 5. Clean job descriptions
# --------------------------------------------

if "description" in jobs_clean.columns:

    jobs_clean["description"] = (
        jobs_clean["description"]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


# --------------------------------------------
# 6. Convert posted date
# --------------------------------------------

if "posted_date" in jobs_clean.columns:

    jobs_clean["posted_date"] = pd.to_datetime(
        jobs_clean["posted_date"],
        errors="coerce",
        utc=True
    )


# --------------------------------------------
# 7. Clean salary columns
# --------------------------------------------

for col in ["salary_min", "salary_max"]:

    if col in jobs_clean.columns:

        jobs_clean[col] = pd.to_numeric(
            jobs_clean[col],
            errors="coerce"
        )


# --------------------------------------------
# 8. Create salary midpoint
# --------------------------------------------

if "salary_min" in jobs_clean.columns and \
   "salary_max" in jobs_clean.columns:

    jobs_clean["salary_mid"] = (
        jobs_clean["salary_min"] +
        jobs_clean["salary_max"]
    ) / 2


# --------------------------------------------
# 9. Create remote indicator
# --------------------------------------------

jobs_clean["is_remote"] = (
    jobs_clean["job_title"]
    .str.lower()
    .str.contains(
        "remote|work from home|wfh",
        regex=True,
        na=False
    )
)


# --------------------------------------------
# 10. Create fresher indicator
# --------------------------------------------

fresher_pattern = (
    r"fresher|freshers|entry.level|"
    r"graduate|0 year|0-1 year|"
    r"junior|trainee|intern"
)

jobs_clean["is_fresher_friendly"] = (
    jobs_clean["job_title"]
    .str.lower()
    .str.contains(
        fresher_pattern,
        regex=True,
        na=False
    )
)


# --------------------------------------------
# 11. Create unique search ID
# --------------------------------------------

jobs_clean["job_unique_key"] = (
    jobs_clean["job_title"].str.lower().str.strip()
    + "_"
    + jobs_clean["company"].str.lower().str.strip()
    + "_"
    + jobs_clean["location"].str.lower().str.strip()
)


# --------------------------------------------
# FINAL DATASET
# --------------------------------------------

jobs_clean = jobs_clean.drop_duplicates(
    subset=["job_unique_key"]
)

jobs_clean = jobs_clean.reset_index(drop=True)


print("\n================================")
print("CLEANING COMPLETE")
print("================================")

print("Final records:", len(jobs_clean))
print("Columns:", len(jobs_clean.columns))

display(jobs_clean.head())

Original records: 250
Duplicates removed: 0

CLEANING COMPLETE
Final records: 231
Columns: 16


,job_id,job_title,company,location,description,posted_date,salary_min,salary_max,job_url,category,search_query,ingested_at,salary_mid,is_remote,is_fresher_friendly,job_unique_key
0,5826247065,Data Analyst,Inficore Soft,India,Data Analyst (Full-Time) Company: Inficore Sof...,2026-08-03 06:51:53+00:00,264000.0,276000.0,https://www.adzuna.in/details/5826247065?utm_m...,IT Jobs,data analyst,2026-10-08T02:13:57.373216+00:00,270000.0,False,False,data analyst_inficore soft_india
1,4686704238,Data Analyst,,"Thiruvananthapuram, Kerala",Immediate requirement for the post of Data Ana...,2024-05-10 07:38:46+00:00,NaN,NaN,https://www.adzuna.in/details/4686704238?utm_m...,IT Jobs,data analyst,2026-10-08T02:13:57.373252+00:00,NaN,False,False,"data analyst__thiruvananthapuram, kerala"
2,5899028729,Data Analyst,ALTRAIZE,"Bangalore, Karnataka",Job Summary We are seeking a detail-oriented a...,2026-09-26 07:08:57+00:00,500000.0,600000.0,https://www.adzuna.in/details/5899028729?utm_m...,IT Jobs,data analyst,2026-10-08T02:13:57.373258+00:00,550000.0,False,False,"data analyst_altraize_bangalore, karnataka"
3,5849116040,Data Analyst,Time Hack Consulting,"Bangalore, Karnataka",Role: Data Analyst / Analytics Engineer About ...,2026-08-20 07:10:41+00:00,600000.0,1300000.0,https://www.adzuna.in/details/5849116040?utm_m...,IT Jobs,data analyst,2026-10-08T02:13:57.373262+00:00,950000.0,False,False,"data analyst_time hack consulting_bangalore, k..."
4,5902287266,Data Analyst,Rekruton Technologies,"Ahmedabad, Gujarat","Role Overview: As a Data Analyst, you turn cam...",2026-09-29 07:58:49+00:00,800000.0,1000000.0,https://www.adzuna.in/details/5902287266?utm_m...,IT Jobs,data analyst,2026-10-08T02:13:57.373266+00:00,900000.0,False,False,"data analyst_rekruton technologies_ahmedabad, ..."


## Cell 10 — jobs_clean.info()


In [ ]:
jobs_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   job_id               231 non-null    object             
 1   job_title            231 non-null    object             
 2   company              231 non-null    object             
 3   location             231 non-null    object             
 4   description          231 non-null    object             
 5   posted_date          231 non-null    datetime64[ns, UTC]
 6   salary_min           105 non-null    float64            
 7   salary_max           105 non-null    float64            
 8   job_url              231 non-null    object             
 9   category             231 non-null    object             
 10  search_query         231 non-null    object             
 11  ingested_at          231 non-null    object             
 12  salary_mid           1

## Cell 11 — display(


In [ ]:
display(
    jobs_clean[
        [
            "job_title",
            "company",
            "location",
            "salary_min",
            "salary_max",
            "is_remote",
            "is_fresher_friendly",
            "search_query"
        ]
    ].head(20)
)

,job_title,company,location,salary_min,salary_max,is_remote,is_fresher_friendly,search_query
0,Data Analyst,Inficore Soft,India,264000.0,276000.0,False,False,data analyst
1,Data Analyst,,"Thiruvananthapuram, Kerala",NaN,NaN,False,False,data analyst
2,Data Analyst,ALTRAIZE,"Bangalore, Karnataka",500000.0,600000.0,False,False,data analyst
3,Data Analyst,Time Hack Consulting,"Bangalore, Karnataka",600000.0,1300000.0,False,False,data analyst
4,Data Analyst,Rekruton Technologies,"Ahmedabad, Gujarat",800000.0,1000000.0,False,False,data analyst
5,Data Analyst,Weekday AI,India,NaN,NaN,False,False,data analyst
6,Data Analyst,Mrsool,India,NaN,NaN,False,False,data analyst
7,Data Analyst,Medico Workforce,India,180000.0,240000.0,False,False,data analyst
8,Data Analyst,Acme Services Private Limited,"Yerwada, Pune",2000000.0,2100000.0,False,False,data analyst
9,Data Analyst,Pratiti Technologies Pvt. Ltd.,"Warje, Pune",NaN,NaN,False,False,data analyst


## Cell 12 — STEP 3: AUTOMATIC SKILL EXTRACTION


In [ ]:
# ============================================
# STEP 3 — AUTOMATIC SKILL EXTRACTION
# ============================================

import re
import pandas as pd

# --------------------------------------------
# Skill dictionary
# --------------------------------------------

SKILLS = {
    "Python": [
        "python",
        "python programming"
    ],

    "SQL": [
        "sql",
        "mysql",
        "postgresql",
        "postgres"
    ],

    "Excel": [
        "excel",
        "microsoft excel",
        "advanced excel"
    ],

    "Power BI": [
        "power bi",
        "powerbi"
    ],

    "Tableau": [
        "tableau"
    ],

    "Looker": [
        "looker",
        "looker studio"
    ],

    "Power Query": [
        "power query"
    ],

    "DAX": [
        "dax"
    ],

    "Pandas": [
        "pandas"
    ],

    "NumPy": [
        "numpy"
    ],

    "Machine Learning": [
        "machine learning",
        "machine-learning"
    ],

    "Deep Learning": [
        "deep learning",
        "deep-learning"
    ],

    "NLP": [
        "natural language processing",
        "nlp"
    ],

    "TensorFlow": [
        "tensorflow"
    ],

    "PyTorch": [
        "pytorch"
    ],

    "Scikit-learn": [
        "scikit-learn",
        "sklearn"
    ],

    "AWS": [
        "aws",
        "amazon web services"
    ],

    "Azure": [
        "azure",
        "microsoft azure"
    ],

    "GCP": [
        "gcp",
        "google cloud"
    ],

    "Git": [
        "git",
        "github",
        "gitlab"
    ],

    "Docker": [
        "docker"
    ],

    "ETL": [
        "etl",
        "extract transform load",
        "extract, transform, load"
    ],

    "Data Visualization": [
        "data visualization",
        "data visualisation"
    ],

    "Statistics": [
        "statistics",
        "statistical analysis"
    ],

    "Data Analysis": [
        "data analysis",
        "data analytics"
    ],

    "Reporting": [
        "reporting",
        "reports"
    ],

    "JIRA": [
        "jira"
    ],

    "SAP": [
        "sap"
    ],

    "Alteryx": [
        "alteryx"
    ]
}


# --------------------------------------------
# Skill extraction function
# --------------------------------------------

def extract_skills(text):

    if pd.isna(text):
        return []

    text = str(text).lower()

    detected_skills = []

    for skill, keywords in SKILLS.items():

        for keyword in keywords:

            # Word-boundary matching
            pattern = r"\b" + re.escape(keyword.lower()) + r"\b"

            if re.search(pattern, text):

                detected_skills.append(skill)
                break

    return detected_skills


# --------------------------------------------
# Apply to job descriptions
# --------------------------------------------

jobs_clean["detected_skills"] = (
    jobs_clean["description"]
    .apply(extract_skills)
)


# --------------------------------------------
# Number of skills per job
# --------------------------------------------

jobs_clean["skill_count"] = (
    jobs_clean["detected_skills"]
    .apply(len)
)


# --------------------------------------------
# Create readable skill string
# --------------------------------------------

jobs_clean["skills"] = (
    jobs_clean["detected_skills"]
    .apply(lambda x: ", ".join(x))
)


print("✅ Skill extraction completed!")

display(
    jobs_clean[
        [
            "job_title",
            "company",
            "skills",
            "skill_count"
        ]
    ].head(20)
)

✅ Skill extraction completed!


,job_title,company,skills,skill_count
0,Data Analyst,Inficore Soft,,0
1,Data Analyst,,,0
2,Data Analyst,ALTRAIZE,,0
3,Data Analyst,Time Hack Consulting,,0
4,Data Analyst,Rekruton Technologies,,0
5,Data Analyst,Weekday AI,"Power BI, GCP, Data Analysis",3
6,Data Analyst,Mrsool,,0
7,Data Analyst,Medico Workforce,"Python, SQL",2
8,Data Analyst,Acme Services Private Limited,,0
9,Data Analyst,Pratiti Technologies Pvt. Ltd.,"Data Analysis, Reporting",2


## Cell 13 — STEP 3: AUTOMATIC SKILL EXTRACTION


In [ ]:
# ============================================
# STEP 3 — AUTOMATIC SKILL EXTRACTION
# ============================================

import re
import pandas as pd

# --------------------------------------------
# Skill dictionary
# --------------------------------------------

SKILLS = {
    "Python": [
        "python",
        "python programming"
    ],

    "SQL": [
        "sql",
        "mysql",
        "postgresql",
        "postgres"
    ],

    "Excel": [
        "excel",
        "microsoft excel",
        "advanced excel"
    ],

    "Power BI": [
        "power bi",
        "powerbi"
    ],

    "Tableau": [
        "tableau"
    ],

    "Looker": [
        "looker",
        "looker studio"
    ],

    "Power Query": [
        "power query"
    ],

    "DAX": [
        "dax"
    ],

    "Pandas": [
        "pandas"
    ],

    "NumPy": [
        "numpy"
    ],

    "Machine Learning": [
        "machine learning",
        "machine-learning"
    ],

    "Deep Learning": [
        "deep learning",
        "deep-learning"
    ],

    "NLP": [
        "natural language processing",
        "nlp"
    ],

    "TensorFlow": [
        "tensorflow"
    ],

    "PyTorch": [
        "pytorch"
    ],

    "Scikit-learn": [
        "scikit-learn",
        "sklearn"
    ],

    "AWS": [
        "aws",
        "amazon web services"
    ],

    "Azure": [
        "azure",
        "microsoft azure"
    ],

    "GCP": [
        "gcp",
        "google cloud"
    ],

    "Git": [
        "git",
        "github",
        "gitlab"
    ],

    "Docker": [
        "docker"
    ],

    "ETL": [
        "etl",
        "extract transform load",
        "extract, transform, load"
    ],

    "Data Visualization": [
        "data visualization",
        "data visualisation"
    ],

    "Statistics": [
        "statistics",
        "statistical analysis"
    ],

    "Data Analysis": [
        "data analysis",
        "data analytics"
    ],

    "Reporting": [
        "reporting",
        "reports"
    ],

    "JIRA": [
        "jira"
    ],

    "SAP": [
        "sap"
    ],

    "Alteryx": [
        "alteryx"
    ]
}


# --------------------------------------------
# Skill extraction function
# --------------------------------------------

def extract_skills(text):

    if pd.isna(text):
        return []

    text = str(text).lower()

    detected_skills = []

    for skill, keywords in SKILLS.items():

        for keyword in keywords:

            # Word-boundary matching
            pattern = r"\b" + re.escape(keyword.lower()) + r"\b"

            if re.search(pattern, text):

                detected_skills.append(skill)
                break

    return detected_skills


# --------------------------------------------
# Apply to job descriptions
# --------------------------------------------

jobs_clean["detected_skills"] = (
    jobs_clean["description"]
    .apply(extract_skills)
)


# --------------------------------------------
# Number of skills per job
# --------------------------------------------

jobs_clean["skill_count"] = (
    jobs_clean["detected_skills"]
    .apply(len)
)


# --------------------------------------------
# Create readable skill string
# --------------------------------------------

jobs_clean["skills"] = (
    jobs_clean["detected_skills"]
    .apply(lambda x: ", ".join(x))
)


print("✅ Skill extraction completed!")

display(
    jobs_clean[
        [
            "job_title",
            "company",
            "skills",
            "skill_count"
        ]
    ].head(20)
)

✅ Skill extraction completed!


,job_title,company,skills,skill_count
0,Data Analyst,Inficore Soft,,0
1,Data Analyst,,,0
2,Data Analyst,ALTRAIZE,,0
3,Data Analyst,Time Hack Consulting,,0
4,Data Analyst,Rekruton Technologies,,0
5,Data Analyst,Weekday AI,"Power BI, GCP, Data Analysis",3
6,Data Analyst,Mrsool,,0
7,Data Analyst,Medico Workforce,"Python, SQL",2
8,Data Analyst,Acme Services Private Limited,,0
9,Data Analyst,Pratiti Technologies Pvt. Ltd.,"Data Analysis, Reporting",2


## Cell 14 — RECREATE SKILL DEMAND ANALYSIS


In [ ]:
# ============================================
# RECREATE SKILL DEMAND ANALYSIS
# ============================================

from collections import Counter

skill_counter = Counter()

for skills in jobs_clean["detected_skills"]:
    skill_counter.update(skills)

skill_demand = pd.DataFrame(
    skill_counter.items(),
    columns=["skill", "job_count"]
)

skill_demand["percentage"] = (
    skill_demand["job_count"]
    / len(jobs_clean)
    * 100
)

skill_demand = skill_demand.sort_values(
    "job_count",
    ascending=False
).reset_index(drop=True)

print("✅ Skill demand analysis created!")

display(skill_demand.head(20))

✅ Skill demand analysis created!


,skill,job_count,percentage
0,Python,59,25.541126
1,Reporting,38,16.450216
2,Data Analysis,33,14.285714
3,SQL,33,14.285714
4,Data Visualization,22,9.523810
5,Machine Learning,18,7.792208
6,Power BI,17,7.359307
7,Statistics,9,3.896104
8,Excel,7,3.030303
9,Tableau,6,2.597403


## Cell 15 — SAVE ENHANCED DATASETS


In [ ]:
# ============================================
# SAVE ENHANCED DATASETS
# ============================================

import os

os.makedirs("data/processed", exist_ok=True)

jobs_clean.to_csv(
    "data/processed/jobs_clean.csv",
    index=False
)

skill_demand.to_csv(
    "data/processed/skill_demand.csv",
    index=False
)

print("✅ jobs_clean.csv saved")
print("✅ skill_demand.csv saved")
print("✅ Step 3 completed successfully!")

✅ jobs_clean.csv saved
✅ skill_demand.csv saved
✅ Step 3 completed successfully!


## Cell 16 — display(skill_demand)


In [ ]:
print("Jobs:", len(jobs_clean))
print("Skills detected:", jobs_clean["skill_count"].sum())
print("Unique skills:", len(skill_demand))

display(skill_demand.head(15))

Jobs: 231
Skills detected: 272
Unique skills: 23


,skill,job_count,percentage
0,Python,59,25.541126
1,Reporting,38,16.450216
2,Data Analysis,33,14.285714
3,SQL,33,14.285714
4,Data Visualization,22,9.523810
5,Machine Learning,18,7.792208
6,Power BI,17,7.359307
7,Statistics,9,3.896104
8,Excel,7,3.030303
9,Tableau,6,2.597403


## Cell 17 — STEP 4: EXPERIENCE EXTRACTION


In [ ]:
# ============================================
# STEP 4 — EXPERIENCE EXTRACTION
# ============================================

import re
import numpy as np

def extract_experience(text):

    if pd.isna(text):
        return {
            "experience_min": np.nan,
            "experience_max": np.nan,
            "experience_text": "Not specified"
        }

    text = str(text).lower()

    # ----------------------------------------
    # Fresher / entry-level indicators
    # ----------------------------------------

    fresher_patterns = [
        r"\bfresher\b",
        r"\bfreshers\b",
        r"\bentry[- ]level\b",
        r"\bno experience\b",
        r"\bwithout experience\b",
        r"\bgraduate\b",
        r"\btrainee\b",
        r"\bintern\b",
        r"\binternship\b"
    ]

    for pattern in fresher_patterns:

        if re.search(pattern, text):

            return {
                "experience_min": 0,
                "experience_max": 1,
                "experience_text": "Fresher / Entry Level"
            }

    # ----------------------------------------
    # Range: 0-1 years
    # ----------------------------------------

    range_pattern = r"(\d+)\s*[-–to]+\s*(\d+)\s*(?:years?|yrs?)"

    match = re.search(range_pattern, text)

    if match:

        minimum = int(match.group(1))
        maximum = int(match.group(2))

        return {
            "experience_min": minimum,
            "experience_max": maximum,
            "experience_text": match.group(0)
        }

    # ----------------------------------------
    # "2+ years"
    # ----------------------------------------

    plus_pattern = r"(\d+)\s*\+\s*(?:years?|yrs?)"

    match = re.search(plus_pattern, text)

    if match:

        minimum = int(match.group(1))

        return {
            "experience_min": minimum,
            "experience_max": np.nan,
            "experience_text": match.group(0)
        }

    # ----------------------------------------
    # "2 years experience"
    # ----------------------------------------

    single_pattern = (
        r"(\d+)\s*(?:years?|yrs?)"
        r"(?:\s*of)?\s*(?:experience|exp)"
    )

    match = re.search(single_pattern, text)

    if match:

        years = int(match.group(1))

        return {
            "experience_min": years,
            "experience_max": years,
            "experience_text": match.group(0)
        }

    # ----------------------------------------
    # No requirement found
    # ----------------------------------------

    return {
        "experience_min": np.nan,
        "experience_max": np.nan,
        "experience_text": "Not specified"
    }


# --------------------------------------------
# Apply function
# --------------------------------------------

experience_data = jobs_clean["description"].apply(
    extract_experience
)

experience_df = pd.DataFrame(
    experience_data.tolist()
)

jobs_clean[
    [
        "experience_min",
        "experience_max",
        "experience_text"
    ]
] = experience_df


print("✅ Experience extraction completed!")

display(
    jobs_clean[
        [
            "job_title",
            "experience_min",
            "experience_max",
            "experience_text"
        ]
    ].head(20)
)

✅ Experience extraction completed!


,job_title,experience_min,experience_max,experience_text
0,Data Analyst,NaN,NaN,Not specified
1,Data Analyst,NaN,NaN,Not specified
2,Data Analyst,NaN,NaN,Not specified
3,Data Analyst,NaN,NaN,Not specified
4,Data Analyst,NaN,NaN,Not specified
5,Data Analyst,4.0,8.0,4–8 years
6,Data Analyst,NaN,NaN,Not specified
7,Data Analyst,NaN,NaN,Not specified
8,Data Analyst,NaN,NaN,Not specified
9,Data Analyst,NaN,NaN,Not specified


## Cell 18 — FRESHER SUITABILITY SCORE


In [ ]:
# ============================================
# FRESHER SUITABILITY SCORE
# ============================================

def calculate_fresher_score(row):

    score = 0

    title = str(row.get("job_title", "")).lower()
    description = str(row.get("description", "")).lower()

    # Explicit fresher indicators
    if re.search(
        r"\bfresher\b|\bfreshers\b|entry[- ]level|"
        r"no experience|graduate|trainee|intern",
        title + " " + description
    ):
        score += 40

    # Experience requirement
    exp_min = row.get("experience_min")

    if pd.isna(exp_min):
        score += 20

    elif exp_min == 0:
        score += 30

    elif exp_min <= 1:
        score += 20

    elif exp_min <= 2:
        score += 10

    # Internship
    if "intern" in title or "internship" in description:
        score += 20

    # Junior
    if "junior" in title:
        score += 10

    # Trainee
    if "trainee" in title:
        score += 10

    return min(score, 100)


jobs_clean["fresher_score"] = jobs_clean.apply(
    calculate_fresher_score,
    axis=1
)


# --------------------------------------------
# Categorize jobs
# --------------------------------------------

def suitability_category(score):

    if score >= 70:
        return "Highly Suitable"

    elif score >= 40:
        return "Potentially Suitable"

    elif score >= 20:
        return "Experience Required"

    else:
        return "Not Suitable"


jobs_clean["fresher_suitability"] = (
    jobs_clean["fresher_score"]
    .apply(suitability_category)
)


print("✅ Fresher suitability scoring completed!")

display(
    jobs_clean[
        [
            "job_title",
            "company",
            "experience_text",
            "fresher_score",
            "fresher_suitability"
        ]
    ]
    .sort_values(
        "fresher_score",
        ascending=False
    )
    .head(20)
)

✅ Fresher suitability scoring completed!


,job_title,company,experience_text,fresher_score,fresher_suitability
143,Data Scientist Data Science Intern,Technobrilliant Learning Solutions,Fresher / Entry Level,90,Highly Suitable
142,Data Scientist Data Science Intern,Kodetocareer,Fresher / Entry Level,90,Highly Suitable
141,Data Scientist Data Science Intern,Amayradh Infotech Private Limited,Fresher / Entry Level,90,Highly Suitable
144,Data Scientist Data Science Intern,Nettech India propmrsarfaraz Ahmed,Fresher / Entry Level,90,Highly Suitable
139,Data Scientist Data Science Intern,Chandigarh University,Fresher / Entry Level,90,Highly Suitable
140,Data Scientist Data Science Intern,Adway Institute Of Management Education Privat...,Fresher / Entry Level,90,Highly Suitable
84,Business Analyst,VDart,0–2 years,70,Highly Suitable
75,Business Analyst,Redfoxa Careerlink Pvt Ltd,Fresher / Entry Level,70,Highly Suitable
114,Data/BI/Business Analyst,Nihilent,Not specified,60,Potentially Suitable
120,BI - Business Intelligence Analyst,FairMoney,Not specified,60,Potentially Suitable


## Cell 19 — STEP 5: JOB MATCH SCORING ENGINE


In [ ]:
# ============================================
# STEP 5 — JOB MATCH SCORING ENGINE
# ============================================

# Skills relevant to a Data Analyst profile
TARGET_SKILLS = {
    "Python",
    "SQL",
    "Excel",
    "Power BI",
    "Tableau",
    "Pandas",
    "NumPy",
    "Statistics",
    "Data Visualization",
    "Data Analysis",
    "DAX",
    "Power Query",
    "ETL"
}


# --------------------------------------------
# 1. Role relevance score — 25 points
# --------------------------------------------

def role_relevance_score(title):

    title = str(title).lower()

    strong_roles = [
        "data analyst",
        "data analytics",
        "bi analyst",
        "business intelligence analyst"
    ]

    related_roles = [
        "business analyst",
        "reporting analyst",
        "analytics analyst",
        "mis analyst"
    ]

    if any(role in title for role in strong_roles):
        return 25

    elif any(role in title for role in related_roles):
        return 20

    elif "data" in title or "analytics" in title:
        return 12

    else:
        return 5


# --------------------------------------------
# 2. Technical skills score — 30 points
# --------------------------------------------

def technical_skill_score(skills):

    if not isinstance(skills, list):
        return 0

    matched = len(
        set(skills).intersection(TARGET_SKILLS)
    )

    # Maximum 30 points
    return min(matched * 5, 30)


# --------------------------------------------
# 3. Data / analytics relevance — 10 points
# --------------------------------------------

def analytics_relevance_score(row):

    text = (
        str(row.get("job_title", "")) + " " +
        str(row.get("description", ""))
    ).lower()

    keywords = [
        "data analysis",
        "data analytics",
        "dashboard",
        "business intelligence",
        "reporting",
        "data visualization",
        "sql",
        "analytics"
    ]

    matches = sum(
        1 for keyword in keywords
        if keyword in text
    )

    return min(matches * 2, 10)


# --------------------------------------------
# 4. Salary score — 5 points
# --------------------------------------------

def salary_score(row):

    salary_min = row.get("salary_min")
    salary_max = row.get("salary_max")

    if pd.notna(salary_min) or pd.notna(salary_max):
        return 5

    return 0


# --------------------------------------------
# 5. Location / work mode score — 5 points
# --------------------------------------------

def location_score(row):

    text = (
        str(row.get("job_title", "")) + " " +
        str(row.get("location", "")) + " " +
        str(row.get("description", ""))
    ).lower()

    if any(
        x in text
        for x in [
            "remote",
            "work from home",
            "hybrid"
        ]
    ):
        return 5

    return 2


# --------------------------------------------
# 6. Calculate final score
# --------------------------------------------

def calculate_job_match(row):

    role_score = role_relevance_score(
        row["job_title"]
    )

    skill_score = technical_skill_score(
        row["detected_skills"]
    )

    fresher_score = (
        row["fresher_score"] * 0.25
    )

    analytics_score = analytics_relevance_score(
        row
    )

    salary = salary_score(row)

    location = location_score(row)

    total = (
        role_score
        + skill_score
        + fresher_score
        + analytics_score
        + salary
        + location
    )

    return round(
        min(total, 100),
        2
    )


jobs_clean["job_match_score"] = jobs_clean.apply(
    calculate_job_match,
    axis=1
)


# --------------------------------------------
# Match category
# --------------------------------------------

def match_category(score):

    if score >= 80:
        return "Excellent Match"

    elif score >= 65:
        return "Strong Match"

    elif score >= 50:
        return "Moderate Match"

    elif score >= 35:
        return "Low Match"

    else:
        return "Poor Match"


jobs_clean["match_category"] = (
    jobs_clean["job_match_score"]
    .apply(match_category)
)


print("✅ Job Match Engine completed!")

✅ Job Match Engine completed!


## Cell 20 — STEP 5.1: IMPROVE JOB MATCH ENGINE


In [ ]:
# ============================================
# STEP 5.1 — IMPROVE JOB MATCH ENGINE
# ============================================

def seniority_penalty(title):

    title = str(title).lower()

    senior_keywords = [
        "senior",
        "sr.",
        "sr ",
        "lead",
        "manager",
        "principal",
        "architect",
        "head",
        "director",
        "specialist"
    ]

    for keyword in senior_keywords:
        if keyword in title:
            return 25

    return 0


def role_level_bonus(title):

    title = str(title).lower()

    # Best fit for our target profile
    if "data analyst" in title:
        return 10

    if "bi analyst" in title:
        return 8

    if "business intelligence" in title:
        return 8

    if "business analyst" in title:
        return 5

    return 0


def improved_job_match(row):

    # Existing components
    role = role_relevance_score(
        row["job_title"]
    )

    skills = technical_skill_score(
        row["detected_skills"]
    )

    fresher = (
        row["fresher_score"] * 0.25
    )

    analytics = analytics_relevance_score(
        row
    )

    salary = salary_score(row)

    location = location_score(row)

    # New components
    seniority = seniority_penalty(
        row["job_title"]
    )

    role_bonus = role_level_bonus(
        row["job_title"]
    )

    total = (
        role
        + skills
        + fresher
        + analytics
        + salary
        + location
        + role_bonus
        - seniority
    )

    return round(
        max(0, min(total, 100)),
        2
    )


jobs_clean["job_match_score"] = jobs_clean.apply(
    improved_job_match,
    axis=1
)


# --------------------------------------------
# Match categories
# --------------------------------------------

jobs_clean["match_category"] = (
    jobs_clean["job_match_score"]
    .apply(match_category)
)


print("✅ Improved matching engine applied!")

✅ Improved matching engine applied!


## Cell 21 — TOP MATCHING JOBS


In [ ]:
# ============================================
# TOP MATCHING JOBS
# ============================================

top_jobs = (
    jobs_clean[
        [
            "job_title",
            "company",
            "location",
            "job_match_score",
            "match_category",
            "fresher_score",
            "skills",
            "job_url"
        ]
    ]
    .sort_values(
        "job_match_score",
        ascending=False
    )
    .head(20)
)

display(top_jobs)

,job_title,company,location,job_match_score,match_category,fresher_score,skills,job_url
31,Data Analyst,Reality Premedia Services Pvt Ltd,"Pune, Maharashtra",78.0,Strong Match,20,"Python, SQL, Power BI, Tableau, Data Analysis",https://www.adzuna.in/details/2500155500?utm_m...
39,Data Analyst,EWU,"Coimbatore, Tamil Nadu",75.0,Strong Match,20,"Power BI, Tableau, Data Visualization, Data An...",https://www.adzuna.in/details/5336689191?utm_m...
24,Data Analyst,Navrang HR Solutions,"Surat, Gujarat",73.0,Strong Match,20,"Python, SQL, Power BI, Looker, Data Analysis, ...",https://www.adzuna.in/details/5727098377?utm_m...
46,Data Analyst,RECEX,"Delhi, India",73.0,Strong Match,20,"SQL, Excel, Power BI, Data Visualization",https://www.adzuna.in/details/5728445327?utm_m...
18,Data Analyst,TechnoGen,"Hyderabad, Telangana",69.0,Strong Match,20,"Python, SQL, Tableau, Looker, ETL, Statistics",https://www.adzuna.in/details/5850375334?utm_m...
37,Data Analyst,Indium Software,India,68.0,Strong Match,20,"Python, SQL, Data Analysis, SAP",https://www.adzuna.in/details/2247134476?utm_m...
19,Data analyst,RECEX,"Kolkata, West Bengal",66.0,Strong Match,60,"Data Visualization, Reporting",https://www.adzuna.in/details/5728436260?utm_m...
10,Data Analyst,Smiths Detection,"Bangalore, Karnataka",66.0,Strong Match,20,"SQL, Power BI, Power Query, DAX, Reporting",https://www.adzuna.in/details/5912750335?utm_m...
42,Data Analyst,Mantras2Success,"Ahmedabad, Gujarat",66.0,Strong Match,20,"Python, SQL, Data Visualization",https://www.adzuna.in/details/5727143467?utm_m...
45,Data Analyst,Syrencloud,"Hyderabad, Telangana",63.0,Moderate Match,20,"SQL, Data Analysis, Reporting, SAP",https://www.adzuna.in/details/2449398766?utm_m...


## Cell 22 — SAVE MATCHED JOB DATA


In [ ]:
# ============================================
# SAVE MATCHED JOB DATA
# ============================================

jobs_clean.to_csv(
    "data/processed/jobs_with_match_scores.csv",
    index=False
)

print("✅ Final job dataset saved!")

✅ Final job dataset saved!


## Cell 23 — STEP 6: CREATE DATABASE


In [ ]:
# ============================================
# STEP 6 — CREATE DATABASE
# ============================================

import sqlite3
import os

os.makedirs("data/database", exist_ok=True)

DB_PATH = "data/database/job_market.db"

conn = sqlite3.connect(DB_PATH)

print("✅ Database created successfully!")
print("Database:", DB_PATH)

✅ Database created successfully!
Database: data/database/job_market.db


## Cell 24 — CREATE JOBS TABLE


In [ ]:
# ============================================
# CREATE JOBS TABLE
# ============================================

create_jobs_table = """
CREATE TABLE IF NOT EXISTS jobs (

    job_id TEXT PRIMARY KEY,

    job_title TEXT,

    company TEXT,

    location TEXT,

    category TEXT,

    description TEXT,

    posted_date TEXT,

    salary_min REAL,

    salary_max REAL,

    salary_mid REAL,

    job_url TEXT,

    search_query TEXT,

    is_remote INTEGER,

    is_fresher_friendly INTEGER,

    experience_min REAL,

    experience_max REAL,

    experience_text TEXT,

    fresher_score REAL,

    fresher_suitability TEXT,

    job_match_score REAL,

    match_category TEXT,

    skills TEXT,

    skill_count INTEGER,

    ingested_at TEXT
);
"""

conn.execute(create_jobs_table)
conn.commit()

print("✅ jobs table created!")

✅ jobs table created!


## Cell 25 — PREPARE DATA FOR DATABASE


In [ ]:
# ============================================
# PREPARE DATA FOR DATABASE
# ============================================

db_jobs = jobs_clean.copy()

# Convert Boolean → Integer
db_jobs["is_remote"] = (
    db_jobs["is_remote"]
    .astype(int)
)

db_jobs["is_fresher_friendly"] = (
    db_jobs["is_fresher_friendly"]
    .astype(int)
)

# Convert datetime to string
db_jobs["posted_date"] = (
    db_jobs["posted_date"]
    .astype(str)
)

# Keep only database columns
db_columns = [
    "job_id",
    "job_title",
    "company",
    "location",
    "category",
    "description",
    "posted_date",
    "salary_min",
    "salary_max",
    "salary_mid",
    "job_url",
    "search_query",
    "is_remote",
    "is_fresher_friendly",
    "experience_min",
    "experience_max",
    "experience_text",
    "fresher_score",
    "fresher_suitability",
    "job_match_score",
    "match_category",
    "skills",
    "skill_count",
    "ingested_at"
]

db_jobs = db_jobs[db_columns]

print("Rows ready:", len(db_jobs))
print("Columns:", len(db_jobs.columns))

Rows ready: 231
Columns: 24


## Cell 26 — INSERT JOB DATA


In [ ]:
# ============================================
# INSERT JOB DATA
# ============================================

db_jobs.to_sql(
    "jobs",
    conn,
    if_exists="replace",
    index=False
)

print("✅ Job data inserted into database!")

✅ Job data inserted into database!


## Cell 27 — query = """


In [ ]:
query = """
SELECT
    job_title,
    company,
    location,
    job_match_score
FROM jobs
ORDER BY job_match_score DESC
LIMIT 10;
"""

top_jobs_sql = pd.read_sql_query(
    query,
    conn
)

display(top_jobs_sql)

,job_title,company,location,job_match_score
0,Data Analyst,Reality Premedia Services Pvt Ltd,"Pune, Maharashtra",78.0
1,Data Analyst,EWU,"Coimbatore, Tamil Nadu",75.0
2,Data Analyst,Navrang HR Solutions,"Surat, Gujarat",73.0
3,Data Analyst,RECEX,"Delhi, India",73.0
4,Data Analyst,TechnoGen,"Hyderabad, Telangana",69.0
5,Data Analyst,Indium Software,India,68.0
6,Data Analyst,Smiths Detection,"Bangalore, Karnataka",66.0
7,Data analyst,RECEX,"Kolkata, West Bengal",66.0
8,Data Analyst,Mantras2Success,"Ahmedabad, Gujarat",66.0
9,Data Analyst,Syrencloud,"Hyderabad, Telangana",63.0


## Cell 28 — query = """


In [ ]:
query = """
SELECT
    company,
    COUNT(*) AS job_count
FROM jobs
WHERE LOWER(job_title) LIKE '%data analyst%'
GROUP BY company
ORDER BY job_count DESC
LIMIT 10;
"""

company_demand = pd.read_sql_query(
    query,
    conn
)

display(company_demand)

,company,job_count
0,Unison Group,2
1,RECEX,2
2,Weekday AI,1
3,Umberto Ceramics International Private Limited,1
4,Time Hack Consulting,1
5,TechnoGen,1
6,Syrencloud,1
7,SveltetechTechnologies Pvt Ltd,1
8,Softcrayons IT Education,1
9,Smiths Detection,1


## Cell 29 — query = """


In [ ]:
query = """
SELECT
    location,
    COUNT(*) AS job_count
FROM jobs
GROUP BY location
ORDER BY job_count DESC
LIMIT 15;
"""

location_demand = pd.read_sql_query(
    query,
    conn
)

display(location_demand)

,location,job_count
0,India,59
1,"Bangalore, Karnataka",50
2,"Pune, Maharashtra",20
3,"Hyderabad, Telangana",20
4,"Noida, Ghaziabad",13
5,"Delhi, India",11
6,"Mumbai, Maharashtra",9
7,"Gurgaon, Haryana",6
8,"Ghaziabad, Uttar Pradesh",4
9,"Coimbatore, Tamil Nadu",4


## Cell 30 — STEP 7.1: MARKET SUMMARY


In [ ]:
# ============================================
# STEP 7.1 — MARKET SUMMARY
# ============================================

query = """
SELECT
    COUNT(*) AS total_jobs,

    COUNT(DISTINCT company) AS unique_companies,

    COUNT(DISTINCT location) AS unique_locations,

    ROUND(AVG(job_match_score), 2) AS avg_match_score,

    ROUND(
        AVG(
            CASE
                WHEN salary_mid IS NOT NULL
                THEN salary_mid
            END
        ),
        2
    ) AS avg_salary
FROM jobs;
"""

market_summary = pd.read_sql_query(
    query,
    conn
)

display(market_summary)

,total_jobs,unique_companies,unique_locations,avg_match_score,avg_salary
0,231,190,34,33.61,999572.17


## Cell 31 — STEP 7.2: SKILL DEMAND


In [ ]:
# ============================================
# STEP 7.2 — SKILL DEMAND
# ============================================

skills_to_analyze = [
    "Python",
    "SQL",
    "Excel",
    "Power BI",
    "Tableau",
    "Machine Learning",
    "Statistics",
    "Data Visualization",
    "Data Analysis",
    "ETL"
]

skill_results = []

for skill in skills_to_analyze:

    query = f"""
    SELECT COUNT(*) AS job_count
    FROM jobs
    WHERE LOWER(skills) LIKE '%{skill.lower()}%';
    """

    count = pd.read_sql_query(
        query,
        conn
    ).iloc[0]["job_count"]

    skill_results.append({
        "skill": skill,
        "job_count": int(count)
    })


sql_skill_demand = pd.DataFrame(
    skill_results
).sort_values(
    "job_count",
    ascending=False
)

display(sql_skill_demand)

,skill,job_count
0,Python,59
1,SQL,33
8,Data Analysis,33
7,Data Visualization,22
5,Machine Learning,18
3,Power BI,17
6,Statistics,9
2,Excel,7
4,Tableau,6
9,ETL,4


## Cell 32 — STEP 7.3: LOCATION DEMAND


In [ ]:
# ============================================
# STEP 7.3 — LOCATION DEMAND
# ============================================

query = """
SELECT
    location,
    COUNT(*) AS job_count
FROM jobs
WHERE location IS NOT NULL
AND location != ''
GROUP BY location
ORDER BY job_count DESC
LIMIT 15;
"""

location_analysis = pd.read_sql_query(
    query,
    conn
)

display(location_analysis)

,location,job_count
0,India,59
1,"Bangalore, Karnataka",50
2,"Pune, Maharashtra",20
3,"Hyderabad, Telangana",20
4,"Noida, Ghaziabad",13
5,"Delhi, India",11
6,"Mumbai, Maharashtra",9
7,"Gurgaon, Haryana",6
8,"Ghaziabad, Uttar Pradesh",4
9,"Coimbatore, Tamil Nadu",4


## Cell 33 — STEP 7.4: FRESHER OPPORTUNITIES


In [ ]:
# ============================================
# STEP 7.4 — FRESHER OPPORTUNITIES
# ============================================

query = """
SELECT
    COUNT(*) AS total_jobs,

    SUM(
        CASE
            WHEN fresher_suitability = 'Highly Suitable'
            THEN 1
            ELSE 0
        END
    ) AS highly_suitable,

    SUM(
        CASE
            WHEN fresher_suitability = 'Potentially Suitable'
            THEN 1
            ELSE 0
        END
    ) AS potentially_suitable,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN fresher_suitability = 'Highly Suitable'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS highly_suitable_percentage

FROM jobs;
"""

fresher_analysis = pd.read_sql_query(
    query,
    conn
)

display(fresher_analysis)

,total_jobs,highly_suitable,potentially_suitable,highly_suitable_percentage
0,231,8,12,3.46


## Cell 34 — STEP 7.5: BEST JOB OPPORTUNITIES


In [ ]:
# ============================================
# STEP 7.5 — BEST JOB OPPORTUNITIES
# ============================================

query = """
SELECT
    job_title,
    company,
    location,
    job_match_score,
    fresher_score,
    fresher_suitability,
    skills,
    job_url

FROM jobs

WHERE job_match_score >= 50

AND (
    LOWER(job_title) LIKE '%data analyst%'
    OR LOWER(job_title) LIKE '%bi analyst%'
    OR LOWER(job_title) LIKE '%business analyst%'
)

ORDER BY
    job_match_score DESC,
    fresher_score DESC

LIMIT 20;
"""

best_jobs = pd.read_sql_query(
    query,
    conn
)

display(best_jobs)

,job_title,company,location,job_match_score,fresher_score,fresher_suitability,skills,job_url
0,Data Analyst,Reality Premedia Services Pvt Ltd,"Pune, Maharashtra",78.0,20,Experience Required,"Python, SQL, Power BI, Tableau, Data Analysis",https://www.adzuna.in/details/2500155500?utm_m...
1,Data Analyst,EWU,"Coimbatore, Tamil Nadu",75.0,20,Experience Required,"Power BI, Tableau, Data Visualization, Data An...",https://www.adzuna.in/details/5336689191?utm_m...
2,Data Analyst,Navrang HR Solutions,"Surat, Gujarat",73.0,20,Experience Required,"Python, SQL, Power BI, Looker, Data Analysis, ...",https://www.adzuna.in/details/5727098377?utm_m...
3,Data Analyst,RECEX,"Delhi, India",73.0,20,Experience Required,"SQL, Excel, Power BI, Data Visualization",https://www.adzuna.in/details/5728445327?utm_m...
4,Data Analyst,TechnoGen,"Hyderabad, Telangana",69.0,20,Experience Required,"Python, SQL, Tableau, Looker, ETL, Statistics",https://www.adzuna.in/details/5850375334?utm_m...
5,Data Analyst,Indium Software,India,68.0,20,Experience Required,"Python, SQL, Data Analysis, SAP",https://www.adzuna.in/details/2247134476?utm_m...
6,Data analyst,RECEX,"Kolkata, West Bengal",66.0,60,Potentially Suitable,"Data Visualization, Reporting",https://www.adzuna.in/details/5728436260?utm_m...
7,Data Analyst,Smiths Detection,"Bangalore, Karnataka",66.0,20,Experience Required,"SQL, Power BI, Power Query, DAX, Reporting",https://www.adzuna.in/details/5912750335?utm_m...
8,Data Analyst,Mantras2Success,"Ahmedabad, Gujarat",66.0,20,Experience Required,"Python, SQL, Data Visualization",https://www.adzuna.in/details/5727143467?utm_m...
9,Data Analyst,Syrencloud,"Hyderabad, Telangana",63.0,20,Experience Required,"SQL, Data Analysis, Reporting, SAP",https://www.adzuna.in/details/2449398766?utm_m...


## Cell 35 — STEP 7.6: SKILL COMBINATIONS


In [ ]:
# ============================================
# STEP 7.6 — SKILL COMBINATIONS
# ============================================

skill_pairs = [
    ("Python", "SQL"),
    ("Python", "Power BI"),
    ("SQL", "Power BI"),
    ("SQL", "Excel"),
    ("Python", "Excel"),
    ("Power BI", "DAX"),
    ("Power BI", "Excel"),
    ("SQL", "Tableau")
]

pair_results = []

for skill1, skill2 in skill_pairs:

    query = f"""
    SELECT COUNT(*) AS job_count
    FROM jobs

    WHERE LOWER(skills) LIKE '%{skill1.lower()}%'
    AND LOWER(skills) LIKE '%{skill2.lower()}%';
    """

    count = pd.read_sql_query(
        query,
        conn
    ).iloc[0]["job_count"]

    pair_results.append({
        "skill_1": skill1,
        "skill_2": skill2,
        "job_count": int(count)
    })


skill_combinations = pd.DataFrame(
    pair_results
).sort_values(
    "job_count",
    ascending=False
)

display(skill_combinations)

,skill_1,skill_2,job_count
0,Python,SQL,17
2,SQL,Power BI,9
7,SQL,Tableau,4
5,Power BI,DAX,4
1,Python,Power BI,2
3,SQL,Excel,1
4,Python,Excel,1
6,Power BI,Excel,1


## Cell 36 — STEP 8.1: CHECK POSTING DATES


In [ ]:
# ============================================
# STEP 8.1 — CHECK POSTING DATES
# ============================================

query = """
SELECT
    job_title,
    company,
    posted_date,
    ingested_at
FROM jobs
ORDER BY posted_date DESC
LIMIT 20;
"""

recent_jobs = pd.read_sql_query(
    query,
    conn
)

display(recent_jobs)

,job_title,company,posted_date,ingested_at
0,Business Analyst,Proof of Skill,2026-10-08 00:43:00+00:00,2026-10-08T02:13:58.410103+00:00
1,Python Developer,RECEX,2026-10-07 21:58:35+00:00,2026-10-08T02:14:00.979119+00:00
2,Analyst-BI & Analytics,Moody's,2026-10-07 19:12:33+00:00,2026-10-08T02:13:59.376459+00:00
3,Data Analyst,Weekday AI,2026-10-07 17:36:34+00:00,2026-10-08T02:13:57.373269+00:00
4,Business Analyst,Innover Systems,2026-10-07 16:37:40+00:00,2026-10-08T02:13:58.410247+00:00
5,Business Analyst,Dynarex Technologies PvtLtd,2026-10-07 16:37:32+00:00,2026-10-08T02:13:58.410250+00:00
6,Data Scientist,Jash Data Sciences Pvt Ltd,2026-10-07 16:37:12+00:00,2026-10-08T02:14:00.234260+00:00
7,Data Scientist,Coface,2026-10-07 04:24:17+00:00,2026-10-08T02:14:00.234225+00:00
8,Python Developer,Trigent Software Private Limited,2026-10-07 00:05:00+00:00,2026-10-08T02:14:00.979104+00:00
9,Data Analyst,Pratiti Technologies Pvt. Ltd.,2026-10-06 18:41:10+00:00,2026-10-08T02:13:57.373284+00:00


## Cell 37 — STEP 8.2: LAST 24 HOURS


In [ ]:
# ============================================
# STEP 8.2 — LAST 24 HOURS
# ============================================

from datetime import datetime, timezone, timedelta

now_utc = datetime.now(timezone.utc)

last_24_hours = now_utc - timedelta(hours=24)

query = """
SELECT
    job_title,
    company,
    location,
    posted_date,
    job_match_score,
    job_url

FROM jobs

WHERE posted_date >= ?

ORDER BY posted_date DESC;
"""

jobs_24h = pd.read_sql_query(
    query,
    conn,
    params=[last_24_hours.isoformat()]
)

print("Jobs posted in last 24 hours:", len(jobs_24h))

display(jobs_24h.head(20))

Jobs posted in last 24 hours: 1


,job_title,company,location,posted_date,job_match_score,job_url
0,Business Analyst,Proof of Skill,"Aperl, Hyderabad",2026-10-08 00:43:00+00:00,39.0,https://www.adzuna.in/details/5917567174?utm_m...


## Cell 38 — STEP 8.3: LAST 7 DAYS


In [ ]:
# ============================================
# STEP 8.3 — LAST 7 DAYS
# ============================================

last_7_days = now_utc - timedelta(days=7)

query = """
SELECT
    COUNT(*) AS jobs_last_7_days
FROM jobs
WHERE posted_date >= ?;
"""

jobs_7d = pd.read_sql_query(
    query,
    conn,
    params=[last_7_days.isoformat()]
)

display(jobs_7d)

,jobs_last_7_days
0,22


## Cell 39 — STEP 8.4: DAILY JOB POSTING TREND


In [ ]:
# ============================================
# STEP 8.4 — DAILY JOB POSTING TREND
# ============================================

query = """
SELECT
    DATE(posted_date) AS posting_date,
    COUNT(*) AS job_count

FROM jobs

WHERE posted_date IS NOT NULL

GROUP BY DATE(posted_date)

ORDER BY posting_date;
"""

daily_trend = pd.read_sql_query(
    query,
    conn
)

display(daily_trend)

,posting_date,job_count
0,2019-06-05,5
1,2020-03-31,1
2,2020-07-04,1
3,2020-09-19,1
4,2020-09-20,1
...,...,...
128,2026-10-03,3
129,2026-10-05,4
130,2026-10-06,3
131,2026-10-07,8


## Cell 40 — STEP 8.5: TOP HIRING COMPANIES


In [ ]:
# ============================================
# STEP 8.5 — TOP HIRING COMPANIES
# ============================================

query = """
SELECT
    company,
    COUNT(*) AS job_count

FROM jobs

WHERE company IS NOT NULL
AND company != ''

GROUP BY company

ORDER BY job_count DESC

LIMIT 15;
"""

top_companies = pd.read_sql_query(
    query,
    conn
)

display(top_companies)

,company,job_count
0,Softcrayons IT Education,5
1,Genpact,4
2,EXL,4
3,Apex 2000,4
4,Vrinda International,3
5,Tiger Analytics,3
6,RECEX,3
7,Ecolab,3
8,Acme Services Private Limited,3
9,Weekday AI,2


## Cell 41 — STEP 8.6: CITY ANALYSIS


In [ ]:
# ============================================
# STEP 8.6 — CITY ANALYSIS
# ============================================

query = """
SELECT
    location,
    COUNT(*) AS job_count

FROM jobs

WHERE (
    LOWER(job_title) LIKE '%data analyst%'
    OR LOWER(job_title) LIKE '%bi analyst%'
    OR LOWER(job_title) LIKE '%business analyst%'
)

GROUP BY location

ORDER BY job_count DESC

LIMIT 20;
"""

city_analysis = pd.read_sql_query(
    query,
    conn
)

display(city_analysis)

,location,job_count
0,India,32
1,"Bangalore, Karnataka",25
2,"Pune, Maharashtra",10
3,"Hyderabad, Telangana",9
4,"Noida, Ghaziabad",6
5,"Delhi, India",5
6,"Gurgaon, Haryana",4
7,"Surat, Gujarat",3
8,"Mumbai, Maharashtra",3
9,"Ahmedabad, Gujarat",3


## Cell 42 — STEP 8.7: DASHBOARD DATASET


In [ ]:
# ============================================
# STEP 8.7 — DASHBOARD DATASET
# ============================================

dashboard_query = """
SELECT
    job_id,
    job_title,
    company,
    location,
    category,

    posted_date,

    salary_min,
    salary_max,
    salary_mid,

    is_remote,
    is_fresher_friendly,

    experience_min,
    experience_max,

    fresher_score,
    fresher_suitability,

    job_match_score,
    match_category,

    skills,
    skill_count,

    search_query,
    job_url,

    ingested_at

FROM jobs;
"""

dashboard_data = pd.read_sql_query(
    dashboard_query,
    conn
)

print("Dashboard records:", len(dashboard_data))
print("Dashboard columns:", len(dashboard_data.columns))

display(dashboard_data.head())

Dashboard records: 231
Dashboard columns: 22


,job_id,job_title,company,location,category,posted_date,salary_min,salary_max,salary_mid,is_remote,...,experience_max,fresher_score,fresher_suitability,job_match_score,match_category,skills,skill_count,search_query,job_url,ingested_at
0,5826247065,Data Analyst,Inficore Soft,India,IT Jobs,2026-08-03 06:51:53+00:00,264000.0,276000.0,270000.0,0,...,NaN,20,Experience Required,47.0,Low Match,,0,data analyst,https://www.adzuna.in/details/5826247065?utm_m...,2026-10-08T02:13:57.373216+00:00
1,4686704238,Data Analyst,,"Thiruvananthapuram, Kerala",IT Jobs,2024-05-10 07:38:46+00:00,NaN,NaN,NaN,0,...,NaN,20,Experience Required,42.0,Low Match,,0,data analyst,https://www.adzuna.in/details/4686704238?utm_m...,2026-10-08T02:13:57.373252+00:00
2,5899028729,Data Analyst,ALTRAIZE,"Bangalore, Karnataka",IT Jobs,2026-09-26 07:08:57+00:00,500000.0,600000.0,550000.0,0,...,NaN,20,Experience Required,47.0,Low Match,,0,data analyst,https://www.adzuna.in/details/5899028729?utm_m...,2026-10-08T02:13:57.373258+00:00
3,5849116040,Data Analyst,Time Hack Consulting,"Bangalore, Karnataka",IT Jobs,2026-08-20 07:10:41+00:00,600000.0,1300000.0,950000.0,0,...,NaN,20,Experience Required,49.0,Low Match,,0,data analyst,https://www.adzuna.in/details/5849116040?utm_m...,2026-10-08T02:13:57.373262+00:00
4,5902287266,Data Analyst,Rekruton Technologies,"Ahmedabad, Gujarat",IT Jobs,2026-09-29 07:58:49+00:00,800000.0,1000000.0,900000.0,0,...,NaN,60,Potentially Suitable,57.0,Moderate Match,,0,data analyst,https://www.adzuna.in/details/5902287266?utm_m...,2026-10-08T02:13:57.373266+00:00


## Cell 43 — SAVE DASHBOARD DATA


In [ ]:
# ============================================
# SAVE DASHBOARD DATA
# ============================================

dashboard_data.to_csv(
    "data/processed/dashboard_jobs.csv",
    index=False
)

print("✅ Dashboard dataset saved!")

✅ Dashboard dataset saved!


## Cell 44 — STEP 9: SCALABLE LIVE JOB INGESTION


In [ ]:
# ============================================
# STEP 9 — SCALABLE LIVE JOB INGESTION
# ============================================

import requests
import time
import json
import os
import pandas as pd
from datetime import datetime, timezone

BASE_URL = "https://api.adzuna.com/v1/api/jobs/in/search"

SEARCH_QUERIES = [
    "data analyst",
    "business analyst",
    "BI analyst",
    "data science",
    "python developer",
    "business intelligence",
    "reporting analyst"
]

RESULTS_PER_PAGE = 50
MAX_PAGES = 5

all_live_jobs = []

for query in SEARCH_QUERIES:

    print("\n" + "=" * 50)
    print(f"SEARCH: {query}")
    print("=" * 50)

    for page in range(1, MAX_PAGES + 1):

        url = f"{BASE_URL}/{page}"

        params = {
            "app_id": ADZUNA_APP_ID,
            "app_key": ADZUNA_APP_KEY,
            "results_per_page": RESULTS_PER_PAGE,
            "what": query,
            "where": "India"
        }

        try:

            response = requests.get(
                url,
                params=params,
                headers={
                    "User-Agent":
                    "Job-Market-Intelligence-Portfolio/1.0"
                },
                timeout=30
            )

            if response.status_code == 200:

                data = response.json()
                jobs = data.get("results", [])

                print(
                    f"Page {page}: "
                    f"{len(jobs)} jobs"
                )

                for job in jobs:

                    job["search_query"] = query
                    job["ingested_at"] = (
                        datetime.now(
                            timezone.utc
                        ).isoformat()
                    )

                    all_live_jobs.append(job)

                # Stop if there are no more results
                if len(jobs) == 0:
                    break

            else:

                print(
                    f"⚠️ Page {page} failed: "
                    f"{response.status_code}"
                )

        except Exception as e:

            print(
                f"❌ Error on page {page}: {e}"
            )

        # Small delay between requests
        time.sleep(1)


print("\n" + "=" * 50)
print("INGESTION FINISHED")
print("=" * 50)

print(
    "Total raw records:",
    len(all_live_jobs)
)


SEARCH: data analyst
Page 1: 50 jobs
Page 2: 50 jobs
Page 3: 50 jobs
Page 4: 50 jobs
Page 5: 50 jobs

SEARCH: business analyst
Page 1: 50 jobs
Page 2: 50 jobs
Page 3: 50 jobs
Page 4: 50 jobs
Page 5: 50 jobs

SEARCH: BI analyst
Page 1: 50 jobs
Page 2: 50 jobs
Page 3: 50 jobs
Page 4: 50 jobs
Page 5: 50 jobs

SEARCH: data science
Page 1: 50 jobs
Page 2: 50 jobs
Page 3: 50 jobs
Page 4: 50 jobs
Page 5: 50 jobs

SEARCH: python developer
Page 1: 50 jobs
Page 2: 50 jobs
Page 3: 50 jobs
Page 4: 50 jobs
Page 5: 50 jobs

SEARCH: business intelligence
Page 1: 50 jobs
Page 2: 50 jobs
Page 3: 50 jobs
Page 4: 50 jobs
Page 5: 50 jobs

SEARCH: reporting analyst
Page 1: 50 jobs
Page 2: 50 jobs
Page 3: 50 jobs
Page 4: 50 jobs
Page 5: 50 jobs

INGESTION FINISHED
Total raw records: 1750


## Cell 45 — SAVE LIVE RAW SNAPSHOT


In [ ]:
# ============================================
# SAVE LIVE RAW SNAPSHOT
# ============================================

os.makedirs("data/raw", exist_ok=True)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

raw_path = (
    f"data/raw/"
    f"adzuna_snapshot_{timestamp}.json"
)

with open(
    raw_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_live_jobs,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ Raw snapshot saved:")
print(raw_path)

✅ Raw snapshot saved:
data/raw/adzuna_snapshot_20261008_021932.json


## Cell 46 — CREATE LIVE DATAFRAME


In [ ]:
# ============================================
# CREATE LIVE DATAFRAME
# ============================================

live_jobs_df = pd.json_normalize(
    all_live_jobs
)

print(
    "Raw records:",
    len(live_jobs_df)
)

print(
    "Columns:",
    len(live_jobs_df.columns)
)

display(
    live_jobs_df.head()
)

Raw records: 1750
Columns: 24


,adref,salary_max,__CLASS__,redirect_url,contract_time,contract_type,title,created,salary_is_predicted,description,...,company.display_name,company.__CLASS__,location.display_name,location.__CLASS__,location.area,category.label,category.__CLASS__,category.tag,latitude,longitude
0,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiaE1ZTmc3N0M4UkdVN...,276000.0,Adzuna::API::Response::Job,https://www.adzuna.in/details/5826247065?utm_m...,full_time,permanent,Data Analyst,2026-08-03T06:51:53Z,0,Data Analyst (Full-Time) Company: Inficore Sof...,...,Inficore Soft,Adzuna::API::Response::Company,India,Adzuna::API::Response::Location,[India],IT Jobs,Adzuna::API::Response::Category,it-jobs,NaN,NaN
1,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNDY4NjcwNDIzOCIsI...,NaN,Adzuna::API::Response::Job,https://www.adzuna.in/details/4686704238?utm_m...,NaN,NaN,Data Analyst,2024-05-10T07:38:46Z,0,Immediate requirement for the post of Data Ana...,...,NaN,Adzuna::API::Response::Company,"Thiruvananthapuram, Kerala",Adzuna::API::Response::Location,"[India, Kerala, Thiruvananthapuram]",IT Jobs,Adzuna::API::Response::Category,it-jobs,8.38159,77.00370
2,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiaE1ZTmc3N0M4UkdVN...,600000.0,Adzuna::API::Response::Job,https://www.adzuna.in/details/5899028729?utm_m...,full_time,permanent,Data Analyst,2026-09-26T07:08:57Z,0,Job Summary We are seeking a detail-oriented a...,...,ALTRAIZE,Adzuna::API::Response::Company,"Bangalore, Karnataka",Adzuna::API::Response::Location,"[India, Karnataka, Bangalore]",IT Jobs,Adzuna::API::Response::Category,it-jobs,NaN,NaN
3,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiaE1ZTmc3N0M4UkdVN...,1300000.0,Adzuna::API::Response::Job,https://www.adzuna.in/details/5849116040?utm_m...,full_time,permanent,Data Analyst,2026-08-20T07:10:41Z,0,Role: Data Analyst / Analytics Engineer About ...,...,Time Hack Consulting,Adzuna::API::Response::Company,"Bangalore, Karnataka",Adzuna::API::Response::Location,"[India, Karnataka, Bangalore]",IT Jobs,Adzuna::API::Response::Category,it-jobs,NaN,NaN
4,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTkwMjI4NzI2NiIsI...,1000000.0,Adzuna::API::Response::Job,https://www.adzuna.in/details/5902287266?utm_m...,full_time,permanent,Data Analyst,2026-09-29T07:58:49Z,0,"Role Overview: As a Data Analyst, you turn cam...",...,Rekruton Technologies,Adzuna::API::Response::Company,"Ahmedabad, Gujarat",Adzuna::API::Response::Location,"[India, Gujarat, Ahmedabad]",IT Jobs,Adzuna::API::Response::Category,it-jobs,23.04200,72.59465


## Cell 47 — DEDUPLICATION


In [ ]:
# ============================================
# DEDUPLICATION
# ============================================

before = len(live_jobs_df)

# Primary deduplication
if "id" in live_jobs_df.columns:

    live_jobs_df = live_jobs_df.drop_duplicates(
        subset=["id"]
    )

# Secondary URL-based deduplication
if "redirect_url" in live_jobs_df.columns:

    live_jobs_df = live_jobs_df.drop_duplicates(
        subset=["redirect_url"]
    )

after = len(live_jobs_df)

print("Records before:", before)
print("Records after:", after)
print("Duplicates removed:", before - after)

Records before: 1750
Records after: 1722
Duplicates removed: 28


## Cell 48 — Compare Previous and New Live Datasets


In [ ]:
print("Previous dataset:", len(jobs_clean))
print("New live dataset:", len(live_jobs_df))

if len(live_jobs_df) > len(jobs_clean):
    print("🚀 Dataset successfully expanded!")
else:
    print("⚠️ Dataset did not expand significantly.")

Previous dataset: 231
New live dataset: 1722
🚀 Dataset successfully expanded!


## Cell 49 — DB_PATH = "data/database/job_market.db"


In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, timezone

DB_PATH = "data/database/job_market.db"

conn = sqlite3.connect(DB_PATH)

# Check whether historical_jobs already exists
tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

print(tables)

              name
0  historical_jobs
1             jobs


## Cell 50 — create_historical_table

In [ ]:
create_historical_table = """
CREATE TABLE IF NOT EXISTS historical_jobs (
    job_id TEXT PRIMARY KEY,
    job_title TEXT,
    company TEXT,
    location TEXT,
    category TEXT,
    description TEXT,
    posted_date TEXT,
    salary_min REAL,
    salary_max REAL,
    salary_mid REAL,
    job_url TEXT,
    search_query TEXT,

    is_remote INTEGER,
    is_fresher_friendly INTEGER,

    experience_min REAL,
    experience_max REAL,
    experience_text TEXT,

    fresher_score REAL,
    fresher_suitability TEXT,

    job_match_score REAL,
    match_category TEXT,

    skills TEXT,
    skill_count INTEGER,

    first_seen_at TEXT,
    last_seen_at TEXT,
    is_active INTEGER,

    source TEXT
);
"""

conn.execute(create_historical_table)
conn.commit()

print("✅ Historical job table created successfully.")

✅ Historical job table created successfully.


## Cell 51 — now = datetime.now(timezone.utc).isoformat()


In [ ]:
# ============================================================
# CELL 51 — BUILD AND UPDATE HISTORICAL JOB DATABASE
# Creates a duplicate-safe historical job tracking system.
# ============================================================

import sqlite3
import pandas as pd
from datetime import datetime, timezone

DB_PATH = "data/database/job_market.db"

# ------------------------------------------------------------
# 1. Connect to SQLite
# ------------------------------------------------------------

conn = sqlite3.connect(DB_PATH)

# ------------------------------------------------------------
# 2. Create historical_jobs table if it doesn't exist
# ------------------------------------------------------------

create_historical_table = """
CREATE TABLE IF NOT EXISTS historical_jobs (
    job_id TEXT PRIMARY KEY,
    job_title TEXT,
    company TEXT,
    location TEXT,
    category TEXT,
    description TEXT,
    posted_date TEXT,
    salary_min REAL,
    salary_max REAL,
    salary_mid REAL,
    job_url TEXT,
    search_query TEXT,

    is_remote INTEGER,
    is_fresher_friendly INTEGER,

    experience_min REAL,
    experience_max REAL,
    experience_text TEXT,

    fresher_score REAL,
    fresher_suitability TEXT,

    job_match_score REAL,
    match_category TEXT,

    skills TEXT,
    skill_count INTEGER,

    first_seen_at TEXT,
    last_seen_at TEXT,
    is_active INTEGER,

    source TEXT
);
"""

conn.execute(create_historical_table)
conn.commit()

print("✅ Historical table ready.")

# ------------------------------------------------------------
# 3. Load the current cleaned/scored dataset
# ------------------------------------------------------------

current_jobs = jobs_clean.copy()

# Support both possible ID column names
if "job_id" not in current_jobs.columns and "id" in current_jobs.columns:
    current_jobs = current_jobs.rename(
        columns={"id": "job_id"}
    )

if "job_id" not in current_jobs.columns:
    raise ValueError(
        "❌ job_id column is missing from jobs_clean."
    )

# ------------------------------------------------------------
# 4. Convert job IDs to strings
# ------------------------------------------------------------

current_jobs["job_id"] = (
    current_jobs["job_id"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 5. Remove duplicate jobs BEFORE database insertion
# ------------------------------------------------------------

before = len(current_jobs)

current_jobs = current_jobs.drop_duplicates(
    subset=["job_id"],
    keep="first"
).copy()

after = len(current_jobs)

print(f"Jobs loaded: {before:,}")
print(f"Duplicate jobs removed: {before - after:,}")
print(f"Unique current jobs: {after:,}")

# ------------------------------------------------------------
# 6. Timestamp
# ------------------------------------------------------------

now = datetime.now(timezone.utc).isoformat()

# ------------------------------------------------------------
# 7. Get existing job IDs from historical database
# ------------------------------------------------------------

existing_ids = set(
    pd.read_sql(
        "SELECT job_id FROM historical_jobs",
        conn
    )["job_id"]
    .astype(str)
)

# ------------------------------------------------------------
# 8. Separate new and existing jobs
# ------------------------------------------------------------

new_jobs = current_jobs[
    ~current_jobs["job_id"].isin(existing_ids)
].copy()

existing_jobs = current_jobs[
    current_jobs["job_id"].isin(existing_ids)
].copy()

print(f"🆕 New jobs: {len(new_jobs):,}")
print(f"🔄 Existing jobs: {len(existing_jobs):,}")

# ------------------------------------------------------------
# 9. Columns required by historical_jobs
# ------------------------------------------------------------

historical_columns = [
    "job_id",
    "job_title",
    "company",
    "location",
    "category",
    "description",
    "posted_date",
    "salary_min",
    "salary_max",
    "salary_mid",
    "job_url",
    "search_query",
    "is_remote",
    "is_fresher_friendly",
    "experience_min",
    "experience_max",
    "experience_text",
    "fresher_score",
    "fresher_suitability",
    "job_match_score",
    "match_category",
    "skills",
    "skill_count",
    "first_seen_at",
    "last_seen_at",
    "is_active",
    "source"
]

# ------------------------------------------------------------
# 10. Insert NEW jobs
# ------------------------------------------------------------

if len(new_jobs) > 0:

    new_jobs["first_seen_at"] = now
    new_jobs["last_seen_at"] = now
    new_jobs["is_active"] = 1
    new_jobs["source"] = "Adzuna"

    new_jobs = new_jobs[
        [
            c for c in historical_columns
            if c in new_jobs.columns
        ]
    ]

    new_jobs.to_sql(
        "historical_jobs",
        conn,
        if_exists="append",
        index=False
    )

    print(
        f"✅ Inserted {len(new_jobs):,} new jobs."
    )

else:

    print("ℹ️ No new jobs to insert.")

# ------------------------------------------------------------
# 11. Update EXISTING jobs
# ------------------------------------------------------------

if len(existing_jobs) > 0:

    for job_id in existing_jobs["job_id"]:

        conn.execute(
            """
            UPDATE historical_jobs
            SET
                last_seen_at = ?,
                is_active = 1
            WHERE job_id = ?
            """,
            (now, job_id)
        )

    conn.commit()

    print(
        f"🔄 Updated {len(existing_jobs):,} existing jobs."
    )

# ------------------------------------------------------------
# 12. Commit changes
# ------------------------------------------------------------

conn.commit()

# ------------------------------------------------------------
# 13. Verify database
# ------------------------------------------------------------

verification = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_historical_jobs,
        COUNT(DISTINCT job_id) AS unique_job_ids,
        COUNT(DISTINCT company) AS companies,
        COUNT(DISTINCT location) AS locations,
        SUM(is_active) AS active_jobs
    FROM historical_jobs
    """,
    conn
)

display(verification)

# ------------------------------------------------------------
# 14. Verify duplicate IDs
# ------------------------------------------------------------

duplicates = pd.read_sql(
    """
    SELECT
        job_id,
        COUNT(*) AS duplicate_count
    FROM historical_jobs
    GROUP BY job_id
    HAVING COUNT(*) > 1
    """,
    conn
)

if len(duplicates) == 0:
    print("✅ Duplicate job IDs: 0")
else:
    print("⚠️ Duplicate job IDs detected:")
    display(duplicates)

# ------------------------------------------------------------
# 15. Close connection
# ------------------------------------------------------------

conn.close()

print("\n" + "=" * 60)
print("✅ HISTORICAL DATABASE UPDATE COMPLETE")
print("=" * 60)

✅ Historical table ready.
Jobs loaded: 231
Duplicate jobs removed: 0
Unique current jobs: 231
🆕 New jobs: 0
🔄 Existing jobs: 231
ℹ️ No new jobs to insert.
🔄 Updated 231 existing jobs.


,total_historical_jobs,unique_job_ids,companies,locations,active_jobs
0,231,231,190,34,231


✅ Duplicate job IDs: 0

✅ HISTORICAL DATABASE UPDATE COMPLETE


In [ ]:
# ============================================================
# REOPEN DATABASE CONNECTION
# Opens SQLite connection for the upcoming analytics queries.
# ============================================================

import sqlite3

DB_PATH = "data/database/job_market.db"

conn = sqlite3.connect(DB_PATH)

print("✅ Database connection reopened successfully.")

✅ Database connection reopened successfully.


## Cell 52 — result = pd.read_sql


In [ ]:
result = pd.read_sql("""
SELECT
    COUNT(*) AS total_jobs,
    COUNT(DISTINCT company) AS companies,
    COUNT(DISTINCT location) AS locations,
    MIN(first_seen_at) AS earliest_seen,
    MAX(last_seen_at) AS latest_seen
FROM historical_jobs
""", conn)

display(result)

,total_jobs,companies,locations,earliest_seen,latest_seen
0,231,190,34,2026-10-08T02:06:49.732973+00:00,2026-10-08T02:31:47.583836+00:00


## Cell 53 — DB_PATH = "data/database/job_market.db"


In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, timezone

DB_PATH = "data/database/job_market.db"

conn = sqlite3.connect(DB_PATH)

# Load current live dataset
current_jobs = jobs_clean.copy()

# Make sure job_id exists
if "job_id" not in current_jobs.columns and "id" in current_jobs.columns:
    current_jobs = current_jobs.rename(columns={"id": "job_id"})

now = datetime.now(timezone.utc).isoformat()

print("Current live jobs:", len(current_jobs))

# Existing historical job IDs
existing_ids = set(
    pd.read_sql(
        "SELECT job_id FROM historical_jobs",
        conn
    )["job_id"].astype(str)
)

current_jobs["job_id"] = current_jobs["job_id"].astype(str)

# Separate new and existing jobs
new_jobs = current_jobs[
    ~current_jobs["job_id"].isin(existing_ids)
].copy()

existing_jobs = current_jobs[
    current_jobs["job_id"].isin(existing_ids)
].copy()

print("🆕 New jobs:", len(new_jobs))
print("🔄 Existing jobs:", len(existing_jobs))

Current live jobs: 231
🆕 New jobs: 0
🔄 Existing jobs: 231


## Cell 54 — new_jobs["first_seen_at"] = now


In [ ]:
if len(new_jobs) > 0:

    new_jobs["first_seen_at"] = now
    new_jobs["last_seen_at"] = now
    new_jobs["is_active"] = 1
    new_jobs["source"] = "Adzuna"

    new_jobs = new_jobs[
        [c for c in historical_columns if c in new_jobs.columns]
    ]

    new_jobs.to_sql(
        "historical_jobs",
        conn,
        if_exists="append",
        index=False
    )

    print(f"✅ Added {len(new_jobs)} new jobs.")

else:
    print("ℹ️ No new jobs found.")

ℹ️ No new jobs found.


## Cell 55 — conn.execute


In [ ]:
if len(existing_jobs) > 0:

    for job_id in existing_jobs["job_id"]:

        conn.execute("""
            UPDATE historical_jobs
            SET
                last_seen_at = ?,
                is_active = 1
            WHERE job_id = ?
        """, (now, job_id))

    conn.commit()

    print(f"🔄 Updated {len(existing_jobs)} existing jobs.")

🔄 Updated 231 existing jobs.


## Cell 56 — verification


In [ ]:
verification = pd.read_sql("""
SELECT
    COUNT(*) AS total_historical_jobs,
    COUNT(DISTINCT company) AS companies,
    COUNT(DISTINCT location) AS locations,
    SUM(is_active) AS active_jobs
FROM historical_jobs
""", conn)

display(verification)

,total_historical_jobs,companies,locations,active_jobs
0,231,190,34,231


## Cell 57 — DB_PATH = "data/database/job_market.db"


In [ ]:
import sqlite3
import pandas as pd

DB_PATH = "data/database/job_market.db"

conn = sqlite3.connect(DB_PATH)

dashboard_df = pd.read_sql("""
SELECT
    job_id,
    job_title,
    company,
    location,
    category,
    posted_date,

    salary_min,
    salary_max,
    salary_mid,

    is_remote,
    is_fresher_friendly,

    experience_min,
    experience_max,
    experience_text,

    job_url,

    fresher_score,
    fresher_suitability,

    job_match_score,
    match_category,

    skills,
    skill_count,

    first_seen_at,
    last_seen_at,
    is_active,

    source,
    search_query

FROM historical_jobs
WHERE is_active = 1
""", conn)

print("Dashboard records:", len(dashboard_df))
display(dashboard_df.head())

Dashboard records: 231


,job_id,job_title,company,location,category,posted_date,salary_min,salary_max,salary_mid,is_remote,...,fresher_suitability,job_match_score,match_category,skills,skill_count,first_seen_at,last_seen_at,is_active,source,search_query
0,5826247065,Data Analyst,Inficore Soft,India,IT Jobs,2026-08-03 06:51:53+00:00,264000.0,276000.0,270000.0,0,...,Experience Required,47.0,Low Match,,0,2026-10-08T02:06:49.732973+00:00,2026-10-08T02:34:58.065887+00:00,1,Adzuna,data analyst
1,4686704238,Data Analyst,,"Thiruvananthapuram, Kerala",IT Jobs,2024-05-10 07:38:46+00:00,NaN,NaN,NaN,0,...,Experience Required,42.0,Low Match,,0,2026-10-08T02:06:49.732973+00:00,2026-10-08T02:34:58.065887+00:00,1,Adzuna,data analyst
2,5899028729,Data Analyst,ALTRAIZE,"Bangalore, Karnataka",IT Jobs,2026-09-26 07:08:57+00:00,500000.0,600000.0,550000.0,0,...,Experience Required,47.0,Low Match,,0,2026-10-08T02:06:49.732973+00:00,2026-10-08T02:34:58.065887+00:00,1,Adzuna,data analyst
3,5849116040,Data Analyst,Time Hack Consulting,"Bangalore, Karnataka",IT Jobs,2026-08-20 07:10:41+00:00,600000.0,1300000.0,950000.0,0,...,Experience Required,49.0,Low Match,,0,2026-10-08T02:06:49.732973+00:00,2026-10-08T02:34:58.065887+00:00,1,Adzuna,data analyst
4,5902287266,Data Analyst,Rekruton Technologies,"Ahmedabad, Gujarat",IT Jobs,2026-09-29 07:58:49+00:00,800000.0,1000000.0,900000.0,0,...,Potentially Suitable,57.0,Moderate Match,,0,2026-10-08T02:06:49.732973+00:00,2026-10-08T02:34:58.065887+00:00,1,Adzuna,data analyst


## Cell 58 — dashboard


In [ ]:
dashboard_df["posted_date"] = pd.to_datetime(
    dashboard_df["posted_date"],
    errors="coerce",
    utc=True
)

dashboard_df["first_seen_at"] = pd.to_datetime(
    dashboard_df["first_seen_at"],
    errors="coerce",
    utc=True
)

dashboard_df["last_seen_at"] = pd.to_datetime(
    dashboard_df["last_seen_at"],
    errors="coerce",
    utc=True
)

# Date-only column
dashboard_df["posted_day"] = dashboard_df["posted_date"].dt.date

# Experience category
def experience_category(row):
    if pd.isna(row["experience_min"]):
        return "Not Specified"
    elif row["experience_min"] == 0:
        return "Fresher"
    elif row["experience_min"] <= 1:
        return "0-1 Years"
    elif row["experience_min"] <= 2:
        return "1-2 Years"
    elif row["experience_min"] <= 3:
        return "2-3 Years"
    elif row["experience_min"] <= 5:
        return "3-5 Years"
    else:
        return "5+ Years"

dashboard_df["experience_category"] = dashboard_df.apply(
    experience_category,
    axis=1
)

# Match score category
def match_category(score):
    if pd.isna(score):
        return "Not Scored"
    elif score >= 75:
        return "Excellent Match"
    elif score >= 60:
        return "Strong Match"
    elif score >= 45:
        return "Potential Match"
    else:
        return "Low Match"

dashboard_df["match_category"] = dashboard_df["job_match_score"].apply(
    match_category
)

print("✅ Dashboard columns created.")

✅ Dashboard columns created.


## Cell 59 — skill_rows

In [ ]:
skill_rows = []

for _, row in dashboard_df.iterrows():

    if pd.isna(row["skills"]):
        continue

    skills = str(row["skills"]).split(",")

    for skill in skills:

        skill = skill.strip()

        if skill:
            skill_rows.append({
                "job_id": row["job_id"],
                "skill": skill
            })

job_skills_df = pd.DataFrame(skill_rows)

print("Job-skill records:", len(job_skills_df))
display(job_skills_df.head())

Job-skill records: 272


,job_id,skill
0,5916983454,Power BI
1,5916983454,GCP
2,5916983454,Data Analysis
3,5891226525,Python
4,5891226525,SQL


## Cell 60 — skill_demand_dashboard


In [ ]:
skill_demand_dashboard = (
    job_skills_df
    .groupby("skill")
    .agg(
        jobs_demand=("job_id", "nunique")
    )
    .reset_index()
    .sort_values(
        "jobs_demand",
        ascending=False
    )
)

total_dashboard_jobs = dashboard_df["job_id"].nunique()

skill_demand_dashboard["demand_percentage"] = (
    skill_demand_dashboard["jobs_demand"]
    / total_dashboard_jobs
    * 100
)

display(skill_demand_dashboard.head(20))

,skill,jobs_demand,demand_percentage
16,Python,59,25.541126
17,Reporting,38,16.450216
3,Data Analysis,33,14.285714
19,SQL,33,14.285714
4,Data Visualization,22,9.523810
12,Machine Learning,18,7.792208
14,Power BI,17,7.359307
20,Statistics,9,3.896104
7,Excel,7,3.030303
21,Tableau,6,2.597403


## Cell 61 — POWER BI DATASET EXPORT — INCLUDING JOB URL


In [ ]:
# ============================================
# POWER BI DATASET EXPORT — INCLUDING JOB URL
# ============================================

import os
import pandas as pd

os.makedirs("data/processed/powerbi", exist_ok=True)

# Make sure job_url exists in the Power BI dataset.
# It comes directly from the historical_jobs table.
if "job_url" not in dashboard_df.columns:
    raise ValueError(
        "job_url is missing from dashboard_df. "
        "Check the historical_jobs query before exporting."
    )

# Clean and validate URLs
dashboard_df["job_url"] = (
    dashboard_df["job_url"]
    .fillna("")
    .astype(str)
    .str.strip()
)

dashboard_df.to_csv(
    "data/processed/powerbi/jobs_dashboard.csv",
    index=False
)

job_skills_df.to_csv(
    "data/processed/powerbi/job_skills.csv",
    index=False
)

skill_demand_dashboard.to_csv(
    "data/processed/powerbi/skill_demand.csv",
    index=False
)

print("✅ Power BI datasets exported successfully.")
print("✅ job_url included in jobs_dashboard.csv")
print("Power BI jobs:", len(dashboard_df))
print("Columns:", len(dashboard_df.columns))

display(
    dashboard_df[
        ["job_id", "job_title", "company", "job_url"]
    ].head(10)
)


✅ Power BI datasets exported successfully.
✅ job_url included in jobs_dashboard.csv
Power BI jobs: 231
Columns: 28


,job_id,job_title,company,job_url
0,5826247065,Data Analyst,Inficore Soft,https://www.adzuna.in/details/5826247065?utm_m...
1,4686704238,Data Analyst,,https://www.adzuna.in/details/4686704238?utm_m...
2,5899028729,Data Analyst,ALTRAIZE,https://www.adzuna.in/details/5899028729?utm_m...
3,5849116040,Data Analyst,Time Hack Consulting,https://www.adzuna.in/details/5849116040?utm_m...
4,5902287266,Data Analyst,Rekruton Technologies,https://www.adzuna.in/details/5902287266?utm_m...
5,5916983454,Data Analyst,Weekday AI,https://www.adzuna.in/details/5916983454?utm_m...
6,5912922283,Data Analyst,Mrsool,https://www.adzuna.in/details/5912922283?utm_m...
7,5891226525,Data Analyst,Medico Workforce,https://www.adzuna.in/details/5891226525?utm_m...
8,5903008902,Data Analyst,Acme Services Private Limited,https://www.adzuna.in/details/5903008902?utm_m...
9,5913832201,Data Analyst,Pratiti Technologies Pvt. Ltd.,https://www.adzuna.in/details/5913832201?utm_m...


**Dataset Download**

## Cell 63 — shutil.make_archive


In [ ]:
# ============================================================
# DOWNLOAD POWER BI DATASETS AS ZIP
# Packages the Power BI files from the current project folder.
# ============================================================

from google.colab import files
from pathlib import Path
import shutil

# Current Power BI folder
POWERBI_FOLDER = Path("data/processed/powerbi")

# ------------------------------------------------------------
# Check that the folder exists
# ------------------------------------------------------------

if not POWERBI_FOLDER.exists():
    raise FileNotFoundError(
        f"❌ Power BI folder not found: {POWERBI_FOLDER.resolve()}"
    )

# ------------------------------------------------------------
# Find Power BI files
# ------------------------------------------------------------

powerbi_files = [
    f for f in POWERBI_FOLDER.iterdir()
    if f.is_file()
]

if not powerbi_files:
    raise FileNotFoundError(
        f"❌ No files found inside {POWERBI_FOLDER}"
    )

print("📁 Power BI files found:\n")

for file in powerbi_files:
    print(f"✅ {file.name}")

# ------------------------------------------------------------
# Create ZIP
# ------------------------------------------------------------

zip_path = shutil.make_archive(
    "PowerBI_Datasets",
    "zip",
    root_dir=POWERBI_FOLDER
)

print("\n" + "=" * 60)
print("✅ POWER BI DATASETS ZIP CREATED")
print("=" * 60)

print(f"ZIP file: {zip_path}")

# ------------------------------------------------------------
# Download ZIP
# ------------------------------------------------------------

files.download(zip_path)

📁 Power BI files found:

✅ skill_demand.csv
✅ job_skills.csv
✅ jobs_dashboard.csv

✅ POWER BI DATASETS ZIP CREATED
ZIP file: /content/PowerBI_Datasets.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# VERIFY PROJECT OUTPUTS
# Checks that all important generated files exist.
# ============================================================

from pathlib import Path

print("=" * 60)
print("PROJECT OUTPUT VERIFICATION")
print("=" * 60)

files_to_check = [
    "data/raw",
    "data/processed",
    "data/processed/powerbi",
    "data/database/job_market.db",
    "data/processed/jobs_clean.csv",
    "data/processed/jobs_with_match_scores.csv",
    "data/processed/dashboard_jobs.csv",
    "data/processed/powerbi/jobs_dashboard.csv",
    "data/processed/powerbi/job_skills.csv",
    "data/processed/powerbi/skill_demand.csv",
]

for item in files_to_check:

    path = Path(item)

    if path.exists():

        if path.is_dir():
            print(f"📁 EXISTS: {item}")

        else:
            size_mb = path.stat().st_size / (1024 * 1024)
            print(f"✅ EXISTS: {item} ({size_mb:.2f} MB)")

    else:
        print(f"❌ MISSING: {item}")

print("\n" + "=" * 60)
print("VERIFICATION COMPLETE")
print("=" * 60)

PROJECT OUTPUT VERIFICATION
📁 EXISTS: data/raw
📁 EXISTS: data/processed
📁 EXISTS: data/processed/powerbi
✅ EXISTS: data/database/job_market.db (0.43 MB)
✅ EXISTS: data/processed/jobs_clean.csv (0.18 MB)
✅ EXISTS: data/processed/jobs_with_match_scores.csv (0.20 MB)
✅ EXISTS: data/processed/dashboard_jobs.csv (0.07 MB)
✅ EXISTS: data/processed/powerbi/jobs_dashboard.csv (0.08 MB)
✅ EXISTS: data/processed/powerbi/job_skills.csv (0.01 MB)
✅ EXISTS: data/processed/powerbi/skill_demand.csv (0.00 MB)

VERIFICATION COMPLETE


## Cell 64 — Ignore system/Colab folders


In [ ]:
import os

for root, dirs, files in os.walk("."):
    # Ignore system/Colab folders
    dirs[:] = [
        d for d in dirs
        if d not in [".config", ".cache", "sample_data", "__pycache__"]
    ]

    level = root.count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

## Cell 65 — JOB MARKET INTELLIGENCE — PROJECT PACKAGING


In [ ]:
import os
import shutil
from pathlib import Path

# ============================================================
# JOB MARKET INTELLIGENCE — PROJECT PACKAGING
# ============================================================

PROJECT = Path("Job-Market-Intelligence")

# ------------------------------------------------------------
# 1. Create complete project structure
# ------------------------------------------------------------

folders = [
    "data/raw",
    "data/processed/powerbi",
    "data/sample",
    "data/database",

    "ingestion",
    "pipeline",
    "database/queries",
    "analytics",
    "dashboard",
    "app",
    "automation",
    "tests",
    "notebooks"
]

for folder in folders:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

print("✅ Project folders created")


# ------------------------------------------------------------
# 2. Copy existing data
# ------------------------------------------------------------

SOURCE_DATA = Path("data")

# Raw files
for file in SOURCE_DATA.glob("raw/*"):
    if file.is_file():
        shutil.copy2(file, PROJECT / "data/raw" / file.name)

# Processed files
for file in SOURCE_DATA.glob("processed/*"):
    if file.is_file():
        shutil.copy2(file, PROJECT / "data/processed" / file.name)

# Power BI files
for file in SOURCE_DATA.glob("processed/powerbi/*"):
    if file.is_file():
        shutil.copy2(file, PROJECT / "data/processed/powerbi" / file.name)

# Database
db_file = SOURCE_DATA / "database" / "job_market.db"

if db_file.exists():
    shutil.copy2(
        db_file,
        PROJECT / "data/database/job_market.db"
    )

print("✅ Existing data and database copied")


# ------------------------------------------------------------
# 3. Create .gitignore
# ------------------------------------------------------------

gitignore = """# Environment variables
.env
*.env

# Python
__pycache__/
*.py[cod]
*.pyo
*.pyd

# Jupyter
.ipynb_checkpoints/

# Virtual environments
venv/
env/
.venv/

# OS
.DS_Store
Thumbs.db

# Secrets
secrets.json
credentials.json

# Local databases
*.sqlite
*.sqlite3

# Temporary files
*.tmp
*.log

# Large generated files
*.parquet

# Keep project structure
!data/**/.gitkeep
"""

(PROJECT / ".gitignore").write_text(
    gitignore,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 4. Create .env.example
# ------------------------------------------------------------

env_example = """# Adzuna API Credentials
# DO NOT put real credentials in this file.

ADZUNA_APP_ID=your_app_id_here
ADZUNA_APP_KEY=your_app_key_here

# Database
DATABASE_PATH=data/database/job_market.db
"""

(PROJECT / ".env.example").write_text(
    env_example,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5. Create requirements.txt
# ------------------------------------------------------------

requirements = """pandas
numpy
requests
python-dotenv
sqlalchemy
psycopg2-binary
"""

(PROJECT / "requirements.txt").write_text(
    requirements,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 6. Create ingestion script
# ------------------------------------------------------------

ingestion_code = r'''
"""
Job Market Intelligence
Live Adzuna Job Ingestion

IMPORTANT:
Store API credentials in environment variables.
Never commit credentials to GitHub.
"""

import os
import json
import time
from datetime import datetime, timezone

import requests
from dotenv import load_dotenv


load_dotenv()

ADZUNA_APP_ID = os.getenv("ADZUNA_APP_ID")
ADZUNA_APP_KEY = os.getenv("ADZUNA_APP_KEY")

BASE_URL = "https://api.adzuna.com/v1/api/jobs/in/search"

SEARCH_QUERIES = [
    "data analyst",
    "business analyst",
    "BI analyst",
    "data science",
    "python developer",
    "business intelligence",
    "reporting analyst"
]

RESULTS_PER_PAGE = 50
MAX_PAGES = 5


def ingest_jobs():

    if not ADZUNA_APP_ID or not ADZUNA_APP_KEY:
        raise ValueError(
            "ADZUNA_APP_ID and ADZUNA_APP_KEY are required."
        )

    all_jobs = []

    for query in SEARCH_QUERIES:

        for page in range(1, MAX_PAGES + 1):

            url = f"{BASE_URL}/{page}"

            params = {
                "app_id": ADZUNA_APP_ID,
                "app_key": ADZUNA_APP_KEY,
                "results_per_page": RESULTS_PER_PAGE,
                "what": query,
                "where": "India"
            }

            response = requests.get(
                url,
                params=params,
                headers={
                    "User-Agent":
                    "Job-Market-Intelligence-Portfolio/1.0"
                },
                timeout=30
            )

            if response.status_code != 200:
                print(
                    f"Request failed: {response.status_code}"
                )
                continue

            data = response.json()

            jobs = data.get("results", [])

            if not jobs:
                break

            for job in jobs:

                job["search_query"] = query

                job["ingested_at"] = (
                    datetime.now(timezone.utc).isoformat()
                )

                all_jobs.append(job)

            print(
                f"{query} | page {page} | "
                f"{len(jobs)} jobs"
            )

            time.sleep(1)

    timestamp = datetime.now(
        timezone.utc
    ).strftime("%Y%m%d_%H%M%S")

    output = (
        f"data/raw/"
        f"adzuna_snapshot_{timestamp}.json"
    )

    os.makedirs("data/raw", exist_ok=True)

    with open(output, "w", encoding="utf-8") as f:
        json.dump(
            all_jobs,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"\nSaved {len(all_jobs)} jobs to {output}"
    )

    return all_jobs


if __name__ == "__main__":
    ingest_jobs()
'''

(PROJECT / "ingestion/adzuna_ingestion.py").write_text(
    ingestion_code,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 7. Create README
# ------------------------------------------------------------

readme = """# Job Market Intelligence & Career Analytics Platform

An end-to-end data analytics and data engineering project that collects
live job-market data, cleans and enriches job postings, extracts skills
and experience requirements, calculates job suitability scores, stores
historical data, and prepares analytics for Power BI.

## Project Architecture

Adzuna API
    ↓
Live Job Ingestion
    ↓
Raw JSON Snapshots
    ↓
Data Cleaning
    ↓
Skill Extraction
    ↓
Experience Extraction
    ↓
Fresher Suitability
    ↓
Job Match Scoring
    ↓
Historical SQLite Database
    ↓
SQL Analytics
    ↓
Power BI Dashboard

## Main Features

- Live job ingestion
- Multi-page API collection
- Job deduplication
- Data cleaning and normalization
- Skill extraction
- Experience extraction
- Fresher suitability scoring
- Job match scoring
- Historical job tracking
- Incremental updates
- SQL analytics
- Power BI-ready datasets
- Job explorer
- Skill-demand analysis
- Market trend analysis

## Current Data Coverage

The project currently contains a large live job dataset collected
from Adzuna for India across roles including:

- Data Analyst
- Business Analyst
- BI Analyst
- Data Science
- Python Developer
- Business Intelligence
- Reporting Analyst

## Repository Structure

Job-Market-Intelligence/
│
├── data/
│   ├── raw/
│   ├── processed/
│   │   └── powerbi/
│   ├── sample/
│   └── database/
│       └── job_market.db
│
├── ingestion/
├── pipeline/
├── database/
│   └── queries/
├── analytics/
├── dashboard/
├── app/
├── automation/
├── tests/
├── notebooks/
│
├── .env.example
├── .gitignore
├── requirements.txt
└── README.md

## Technologies

- Python
- Pandas
- NumPy
- Requests
- SQLite
- SQL
- Power BI
- DAX
- Git
- GitHub
- Adzuna API

## Security

API credentials must be stored in environment variables.

Never commit `.env` or API keys to GitHub.

## Future Improvements

- PostgreSQL migration
- Automated GitHub Actions ingestion
- Resume-to-job matching
- Streamlit career intelligence application
- Advanced skill normalization
- Job trend forecasting
- Automated Power BI refresh
"""

(PROJECT / "README.md").write_text(
    readme,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 8. Create placeholder files for future modules
# ------------------------------------------------------------

placeholder_files = [
    "pipeline/cleaning.py",
    "pipeline/skill_extraction.py",
    "pipeline/experience_extraction.py",
    "pipeline/job_scoring.py",
    "database/database_manager.py",
    "analytics/sql_analytics.py",
    "automation/daily_pipeline.py",
    "dashboard/README.md",
    "app/README.md",
    "tests/test_pipeline.py"
]

for file in placeholder_files:

    path = PROJECT / file

    if not path.exists():

        path.write_text(
            '"""Module documentation will be added here."""\n',
            encoding="utf-8"
        )


# ------------------------------------------------------------
# 9. Create project inventory
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PROJECT CREATED SUCCESSFULLY")
print("=" * 60)

for root, dirs, files in os.walk(PROJECT):

    level = root.count(os.sep) - PROJECT.as_posix().count("/")

    indent = "    " * max(level, 0)

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

print("\n✅ All existing data preserved.")
print("✅ Database copied.")
print("✅ Raw API snapshots copied.")
print("✅ Power BI datasets copied.")
print("✅ GitHub configuration created.")
print("✅ README created.")

## Cell 66 — PROJECT = Path("Job-Market-Intelligence")


In [ ]:
from pathlib import Path

PROJECT = Path("Job-Market-Intelligence")

# ============================================================
# 1. DATA CLEANING
# ============================================================

cleaning_code = r'''
import pandas as pd
import re


def clean_jobs(df):

    df = df.copy()

    rename_map = {
        "id": "job_id",
        "title": "job_title",
        "company.display_name": "company",
        "location.display_name": "location",
        "category.label": "category",
        "description": "description",
        "redirect_url": "job_url",
        "created": "posted_date"
    }

    df.rename(
        columns={
            k: v for k, v in rename_map.items()
            if k in df.columns
        },
        inplace=True
    )

    # --------------------------------------------------------
    # Basic text cleaning
    # --------------------------------------------------------

    text_columns = [
        "job_title",
        "company",
        "location",
        "category",
        "description"
    ]

    for col in text_columns:

        if col in df.columns:

            df[col] = (
                df[col]
                .fillna("")
                .astype(str)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
            )

    # --------------------------------------------------------
    # Dates
    # --------------------------------------------------------

    if "posted_date" in df.columns:

        df["posted_date"] = pd.to_datetime(
            df["posted_date"],
            errors="coerce",
            utc=True
        )

    # --------------------------------------------------------
    # Salary
    # --------------------------------------------------------

    for col in ["salary_min", "salary_max"]:

        if col in df.columns:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    if "salary_min" in df.columns and "salary_max" in df.columns:

        df["salary_mid"] = (
            df["salary_min"] +
            df["salary_max"]
        ) / 2

    # --------------------------------------------------------
    # Remote
    # --------------------------------------------------------

    df["is_remote"] = (
        df.get("description", "")
        .astype(str)
        .str.contains(
            r"remote|work from home|wfh",
            case=False,
            regex=True
        )
    )

    # --------------------------------------------------------
    # Unique job key
    # --------------------------------------------------------

    df["job_unique_key"] = (
        df["job_title"].str.lower().str.strip()
        + "|"
        + df["company"].str.lower().str.strip()
        + "|"
        + df["location"].str.lower().str.strip()
    )

    # Remove duplicate job IDs
    if "job_id" in df.columns:
        df = df.drop_duplicates(
            subset=["job_id"]
        )

    # Remove duplicate title/company/location
    df = df.drop_duplicates(
        subset=["job_unique_key"]
    )

    # --------------------------------------------------------
    # Fresher-friendly detection
    # --------------------------------------------------------

    fresher_pattern = (
        r"fresher|freshers|entry[- ]level|"
        r"graduate|trainee|intern|internship|"
        r"no experience|0[- ]?1 years?"
    )

    df["is_fresher_friendly"] = (
        df["description"]
        .str.contains(
            fresher_pattern,
            case=False,
            regex=True
        )
    )

    return df


if __name__ == "__main__":

    input_file = (
        "data/processed/"
        "jobs_initial.csv"
    )

    output_file = (
        "data/processed/"
        "jobs_clean.csv"
    )

    df = pd.read_csv(input_file)

    cleaned = clean_jobs(df)

    cleaned.to_csv(
        output_file,
        index=False
    )

    print(
        f"Saved {len(cleaned)} cleaned jobs."
    )
'''

(PROJECT / "pipeline/cleaning.py").write_text(
    cleaning_code,
    encoding="utf-8"
)


# ============================================================
# 2. SKILL EXTRACTION
# ============================================================

skill_code = r'''
import pandas as pd


SKILLS = [
    "Python",
    "SQL",
    "Excel",
    "Power BI",
    "Tableau",
    "Looker",
    "Power Query",
    "DAX",
    "Pandas",
    "NumPy",
    "Machine Learning",
    "Deep Learning",
    "NLP",
    "TensorFlow",
    "PyTorch",
    "Scikit-learn",
    "AWS",
    "Azure",
    "GCP",
    "Git",
    "Docker",
    "ETL",
    "Data Visualization",
    "Statistics",
    "Data Analysis",
    "Reporting",
    "JIRA",
    "SAP",
    "Alteryx"
]


def extract_skills(text):

    text = str(text).lower()

    detected = []

    for skill in SKILLS:

        if skill.lower() in text:
            detected.append(skill)

    return detected


def add_skill_features(df):

    df = df.copy()

    combined_text = (
        df["job_title"].fillna("")
        + " "
        + df["description"].fillna("")
    )

    df["detected_skills"] = (
        combined_text.apply(extract_skills)
    )

    df["skill_count"] = (
        df["detected_skills"]
        .apply(len)
    )

    df["skills"] = (
        df["detected_skills"]
        .apply(lambda x: ", ".join(x))
    )

    return df


if __name__ == "__main__":

    input_file = "data/processed/jobs_clean.csv"
    output_file = "data/processed/jobs_with_skills.csv"

    df = pd.read_csv(input_file)

    df = add_skill_features(df)

    df.to_csv(
        output_file,
        index=False
    )

    print(
        f"Saved {len(df)} jobs with skill extraction."
    )
'''

(PROJECT / "pipeline/skill_extraction.py").write_text(
    skill_code,
    encoding="utf-8"
)


# ============================================================
# 3. EXPERIENCE EXTRACTION
# ============================================================

experience_code = r'''
import re
import pandas as pd


def extract_experience(text):

    text = str(text).lower()

    minimum = None
    maximum = None

    # 0-1 years
    range_match = re.search(
        r"(\d+)\s*[-–]\s*(\d+)\s*years?",
        text
    )

    if range_match:

        minimum = int(range_match.group(1))
        maximum = int(range_match.group(2))

    else:

        plus_match = re.search(
            r"(\d+)\s*\+?\s*years?",
            text
        )

        if plus_match:

            minimum = int(
                plus_match.group(1)
            )

    # Explicit fresher signals
    fresher_terms = [
        "fresher",
        "freshers",
        "entry level",
        "entry-level",
        "no experience",
        "graduate",
        "trainee",
        "intern",
        "internship"
    ]

    has_fresher_signal = any(
        term in text
        for term in fresher_terms
    )

    if has_fresher_signal and minimum is None:

        minimum = 0
        maximum = 0

    return minimum, maximum


def add_experience_features(df):

    df = df.copy()

    results = (
        df["description"]
        .fillna("")
        .apply(extract_experience)
    )

    df["experience_min"] = (
        results.apply(lambda x: x[0])
    )

    df["experience_max"] = (
        results.apply(lambda x: x[1])
    )

    df["experience_text"] = (
        df["experience_min"]
        .fillna("")
        .astype(str)
    )

    return df


def calculate_fresher_score(row):

    score = 0

    title = str(
        row.get("job_title", "")
    ).lower()

    description = str(
        row.get("description", "")
    ).lower()

    minimum = row.get(
        "experience_min"
    )

    # Explicit entry-level indicators
    if any(
        x in description
        for x in [
            "fresher",
            "entry-level",
            "entry level",
            "no experience"
        ]
    ):
        score += 40

    if minimum == 0:
        score += 30

    elif minimum == 1:
        score += 20

    elif minimum == 2:
        score += 10

    if "intern" in description:
        score += 20

    if "junior" in title:
        score += 10

    if "trainee" in description:
        score += 10

    return min(score, 100)


def add_fresher_scores(df):

    df = df.copy()

    df["fresher_score"] = df.apply(
        calculate_fresher_score,
        axis=1
    )

    def suitability(score):

        if score >= 70:
            return "Highly Suitable"

        elif score >= 40:
            return "Potentially Suitable"

        elif score >= 20:
            return "Experience Required"

        return "Not Suitable"

    df["fresher_suitability"] = (
        df["fresher_score"]
        .apply(suitability)
    )

    return df
'''

(PROJECT / "pipeline/experience_extraction.py").write_text(
    experience_code,
    encoding="utf-8"
)


print("✅ Core pipeline scripts created:")
print("   pipeline/cleaning.py")
print("   pipeline/skill_extraction.py")
print("   pipeline/experience_extraction.py")

## Cell 67 — PROJECT = Path("Job-Market-Intelligence")


In [ ]:
from pathlib import Path

PROJECT = Path("Job-Market-Intelligence")

# ============================================================
# 1. JOB MATCH SCORING
# ============================================================

scoring_code = r'''
import pandas as pd


def seniority_penalty(title):

    title = str(title).lower()

    senior_terms = [
        "senior",
        "sr.",
        "sr ",
        "lead",
        "manager",
        "principal",
        "architect",
        "head",
        "director"
    ]

    return 25 if any(
        term in title
        for term in senior_terms
    ) else 0


def role_level_bonus(title):

    title = str(title).lower()

    if "data analyst" in title:
        return 10

    if "bi analyst" in title:
        return 8

    if "business analyst" in title:
        return 5

    return 0


def calculate_job_match(row):

    title = str(
        row.get("job_title", "")
    ).lower()

    skills = str(
        row.get("skills", "")
    ).lower()

    score = 0

    # --------------------------------------------------------
    # Role relevance
    # --------------------------------------------------------

    if "data analyst" in title:
        score += 25

    elif "business analyst" in title:
        score += 22

    elif "bi analyst" in title:
        score += 23

    elif "business intelligence" in title:
        score += 20

    elif "data science" in title:
        score += 18

    elif "python developer" in title:
        score += 15

    # --------------------------------------------------------
    # Technical skills
    # --------------------------------------------------------

    important_skills = [
        "python",
        "sql",
        "power bi",
        "tableau",
        "excel",
        "data analysis",
        "data visualization",
        "statistics"
    ]

    matched = sum(
        skill in skills
        for skill in important_skills
    )

    score += min(
        matched * 4,
        30
    )

    # --------------------------------------------------------
    # Fresher suitability
    # --------------------------------------------------------

    fresher_score = row.get(
        "fresher_score",
        0
    )

    if pd.notna(fresher_score):
        score += (
            float(fresher_score) * 0.25
        )

    # --------------------------------------------------------
    # Analytics relevance
    # --------------------------------------------------------

    analytics_terms = [
        "data analysis",
        "reporting",
        "dashboard",
        "analytics",
        "business intelligence",
        "data visualization"
    ]

    description = str(
        row.get("description", "")
    ).lower()

    analytics_matches = sum(
        term in description
        for term in analytics_terms
    )

    score += min(
        analytics_matches * 2,
        10
    )

    # --------------------------------------------------------
    # Salary
    # --------------------------------------------------------

    salary = row.get(
        "salary_mid"
    )

    if pd.notna(salary) and salary > 0:

        score += 5

    # --------------------------------------------------------
    # Remote/location
    # --------------------------------------------------------

    if row.get("is_remote", False):

        score += 5

    # --------------------------------------------------------
    # Role bonus
    # --------------------------------------------------------

    score += role_level_bonus(title)

    # --------------------------------------------------------
    # Seniority penalty
    # --------------------------------------------------------

    score -= seniority_penalty(title)

    return max(
        0,
        min(
            round(score, 2),
            100
        )
    )


def add_match_scores(df):

    df = df.copy()

    df["job_match_score"] = df.apply(
        calculate_job_match,
        axis=1
    )

    def category(score):

        if score >= 75:
            return "Excellent Match"

        elif score >= 60:
            return "Strong Match"

        elif score >= 45:
            return "Potential Match"

        return "Low Match"

    df["match_category"] = (
        df["job_match_score"]
        .apply(category)
    )

    return df


if __name__ == "__main__":

    input_file = (
        "data/processed/"
        "jobs_with_experience.csv"
    )

    output_file = (
        "data/processed/"
        "jobs_with_match_scores.csv"
    )

    df = pd.read_csv(input_file)

    df = add_match_scores(df)

    df.to_csv(
        output_file,
        index=False
    )

    print(
        f"Saved {len(df)} scored jobs."
    )
'''

(PROJECT / "pipeline/job_scoring.py").write_text(
    scoring_code,
    encoding="utf-8"
)


# ============================================================
# 2. DATABASE MANAGER
# ============================================================

database_code = r'''
import sqlite3
from pathlib import Path
import pandas as pd


DATABASE_PATH = (
    "data/database/job_market.db"
)


def get_connection():

    Path(
        DATABASE_PATH
    ).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    return sqlite3.connect(
        DATABASE_PATH
    )


def create_tables(conn):

    conn.execute("""
    CREATE TABLE IF NOT EXISTS historical_jobs (

        job_id TEXT PRIMARY KEY,

        job_title TEXT,
        company TEXT,
        location TEXT,
        category TEXT,
        description TEXT,

        posted_date TEXT,

        salary_min REAL,
        salary_max REAL,
        salary_mid REAL,

        job_url TEXT,
        search_query TEXT,

        is_remote INTEGER,
        is_fresher_friendly INTEGER,

        experience_min REAL,
        experience_max REAL,
        experience_text TEXT,

        fresher_score REAL,
        fresher_suitability TEXT,

        job_match_score REAL,
        match_category TEXT,

        skills TEXT,
        skill_count INTEGER,

        first_seen_at TEXT,
        last_seen_at TEXT,

        is_active INTEGER,

        source TEXT
    )
    """)

    conn.commit()


def load_jobs(conn):

    return pd.read_sql(
        "SELECT * FROM historical_jobs",
        conn
    )


def insert_jobs(df, conn):

    if df.empty:
        return 0

    df = df.copy()

    existing = pd.read_sql(
        "SELECT job_id FROM historical_jobs",
        conn
    )

    existing_ids = set(
        existing["job_id"].astype(str)
    )

    df["job_id"] = (
        df["job_id"].astype(str)
    )

    new_jobs = df[
        ~df["job_id"].isin(existing_ids)
    ].copy()

    if new_jobs.empty:
        return 0

    new_jobs.to_sql(
        "historical_jobs",
        conn,
        if_exists="append",
        index=False
    )

    return len(new_jobs)


if __name__ == "__main__":

    conn = get_connection()

    create_tables(conn)

    jobs = load_jobs(conn)

    print(
        f"Historical database contains "
        f"{len(jobs)} jobs."
    )

    conn.close()
'''

(PROJECT / "database/database_manager.py").write_text(
    database_code,
    encoding="utf-8"
)


# ============================================================
# 3. SQL ANALYTICS
# ============================================================

analytics_code = r'''
import sqlite3
import pandas as pd


DATABASE_PATH = (
    "data/database/job_market.db"
)


def run_analytics():

    conn = sqlite3.connect(
        DATABASE_PATH
    )

    # --------------------------------------------------------
    # Market summary
    # --------------------------------------------------------

    summary = pd.read_sql("""
        SELECT
            COUNT(*) AS total_jobs,
            COUNT(DISTINCT company) AS companies,
            COUNT(DISTINCT location) AS locations,
            AVG(job_match_score) AS avg_match_score,
            AVG(salary_mid) AS avg_salary
        FROM historical_jobs
        WHERE is_active = 1
    """, conn)

    print("\nMARKET SUMMARY")
    print(summary)

    # --------------------------------------------------------
    # Top companies
    # --------------------------------------------------------

    companies = pd.read_sql("""
        SELECT
            company,
            COUNT(*) AS job_count
        FROM historical_jobs
        WHERE is_active = 1
        GROUP BY company
        ORDER BY job_count DESC
        LIMIT 20
    """, conn)

    print("\nTOP COMPANIES")
    print(companies)

    # --------------------------------------------------------
    # Top locations
    # --------------------------------------------------------

    locations = pd.read_sql("""
        SELECT
            location,
            COUNT(*) AS job_count
        FROM historical_jobs
        WHERE is_active = 1
        GROUP BY location
        ORDER BY job_count DESC
        LIMIT 20
    """, conn)

    print("\nTOP LOCATIONS")
    print(locations)

    # --------------------------------------------------------
    # Fresher opportunities
    # --------------------------------------------------------

    fresher = pd.read_sql("""
        SELECT
            fresher_suitability,
            COUNT(*) AS jobs
        FROM historical_jobs
        WHERE is_active = 1
        GROUP BY fresher_suitability
        ORDER BY jobs DESC
    """, conn)

    print("\nFRESHER OPPORTUNITIES")
    print(fresher)

    # --------------------------------------------------------
    # Best jobs
    # --------------------------------------------------------

    best_jobs = pd.read_sql("""
        SELECT
            job_title,
            company,
            location,
            job_match_score,
            fresher_score,
            skills,
            job_url
        FROM historical_jobs
        WHERE is_active = 1
        ORDER BY job_match_score DESC
        LIMIT 20
    """, conn)

    print("\nBEST MATCHING JOBS")
    print(best_jobs)

    conn.close()

    return {
        "summary": summary,
        "companies": companies,
        "locations": locations,
        "fresher": fresher,
        "best_jobs": best_jobs
    }


if __name__ == "__main__":
    run_analytics()
'''

(PROJECT / "analytics/sql_analytics.py").write_text(
    analytics_code,
    encoding="utf-8"
)


# ============================================================
# 4. MASTER PIPELINE
# ============================================================

master_pipeline = r'''
"""
JOB MARKET INTELLIGENCE
Master Pipeline

This script is the central entry point for the project.

Future production flow:

API
 ↓
Raw ingestion
 ↓
Cleaning
 ↓
Skill extraction
 ↓
Experience extraction
 ↓
Scoring
 ↓
Database
 ↓
Analytics
"""

from pathlib import Path

print("=" * 60)
print("JOB MARKET INTELLIGENCE PIPELINE")
print("=" * 60)

print("""
Pipeline modules:

1. Live ingestion
2. Data cleaning
3. Skill extraction
4. Experience extraction
5. Job scoring
6. Historical database
7. SQL analytics

Run individual modules while development is in progress.
""")

print("Project structure verified.")
'''

(PROJECT / "run_pipeline.py").write_text(
    master_pipeline,
    encoding="utf-8"
)


# ============================================================
# 5. SQL QUERY FILES
# ============================================================

queries = {
    "market_summary.sql": """
SELECT
    COUNT(*) AS total_jobs,
    COUNT(DISTINCT company) AS companies,
    COUNT(DISTINCT location) AS locations,
    AVG(job_match_score) AS avg_match_score
FROM historical_jobs
WHERE is_active = 1;
""",

    "top_companies.sql": """
SELECT
    company,
    COUNT(*) AS job_count
FROM historical_jobs
WHERE is_active = 1
GROUP BY company
ORDER BY job_count DESC
LIMIT 20;
""",

    "top_locations.sql": """
SELECT
    location,
    COUNT(*) AS job_count
FROM historical_jobs
WHERE is_active = 1
GROUP BY location
ORDER BY job_count DESC
LIMIT 20;
""",

    "best_jobs.sql": """
SELECT
    job_title,
    company,
    location,
    skills,
    fresher_score,
    job_match_score,
    job_url
FROM historical_jobs
WHERE is_active = 1
ORDER BY job_match_score DESC
LIMIT 20;
""",

    "fresher_jobs.sql": """
SELECT
    fresher_suitability,
    COUNT(*) AS job_count
FROM historical_jobs
WHERE is_active = 1
GROUP BY fresher_suitability
ORDER BY job_count DESC;
"""
}

for filename, query in queries.items():

    (
        PROJECT /
        "database/queries" /
        filename
    ).write_text(
        query.strip() + "\n",
        encoding="utf-8"
    )


print("\n" + "=" * 60)
print("✅ STAGE 2 COMPLETE")
print("=" * 60)

print("""
Created:

pipeline/
├── job_scoring.py
├── cleaning.py
├── skill_extraction.py
└── experience_extraction.py

database/
├── database_manager.py
└── queries/
    ├── market_summary.sql
    ├── top_companies.sql
    ├── top_locations.sql
    ├── best_jobs.sql
    └── fresher_jobs.sql

analytics/
└── sql_analytics.py

run_pipeline.py
""")